In [1]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [10]:
# Example 2: Manager–Worker Multi-Agent System
# 🧠 Scenario

# “Now instead of fixed flow, we introduce a Manager Agent
# that assigns tasks dynamically to worker

# ================================
# WORKER AGENTS
# ================================
def flight_agent():
    print("[Flight Agent] Working...")
    return [{"airline": "IndiGo", "price": 4500},
            {"airline": "Air India", "price": 5200}]


def weather_agent():
    print("[Weather Agent] Working...")
    return {"condition": "Clear", "temp": 28}


# ================================
# MANAGER AGENT
# ================================
def manager_agent(user_query):
    print("\n[Manager Agent] Analyzing task...")

    tasks = []

    if "trip" in user_query.lower():
        tasks = ["flight", "weather"]

    return tasks


# ================================
# EXECUTION
# ================================
def run_system(user_query):
    print("User Query:", user_query)

    tasks = manager_agent(user_query)

    results = {}

    for task in tasks:
        if task == "flight":
            results["flights"] = flight_agent()

        elif task == "weather":
            results["weather"] = weather_agent()

    # Final decision
    cheapest = min(results["flights"], key=lambda x: x["price"])

    return f"Manager Decision: Book {cheapest['airline']} at ₹{cheapest['price']}"


# RUN
response = run_system("Plan my trip")
print("\nFinal Answer:", response)


User Query: Plan my trip

[Manager Agent] Analyzing task...
[Flight Agent] Working...
[Weather Agent] Working...

Final Answer: Manager Decision: Book IndiGo at ₹4500


In [ ]:
!pip install groq


In [9]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)
# ================================

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions
# ================================

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines
# ================================

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
# ================================

import os
import json
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "openai/gpt-oss-120b"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1200
    )
    return completion.choices[0].message.content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end])
        raise ValueError(f"Model did not return valid JSON. Raw response: {text}")


class IntakeAgent:
    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 1: Intake Agent] Reviewing patient history and planning next steps...")

        system_prompt = (
            "You are Agent 1 in a hospital sequential multi-agent system. "
            "Your job is intake planning only. "
            "Read the patient case, summarize the situation, identify the next steps, "
            "set urgency, and keep the answer practical and structured. "
            "Return only valid JSON with exactly these keys: "
            "summary, required_steps, priority_level, intake_notes. "
            "Keep required_steps short and realistic. "
            "Do not include any markdown, explanation outside JSON, or extra keys."
        )

        user_prompt = f"""
Patient Case:
{json.dumps(patient_data, indent=2)}

Output rules:
- summary: 2 to 3 lines
- required_steps: short list of immediate steps
- priority_level: low, medium, or high
- intake_notes: concise operational notes
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DiagnosticAgent:
    def simulate_tests(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        symptoms = [s.lower() for s in patient_data.get("symptoms", [])]
        vitals = patient_data.get("vitals", {})
        temperature = vitals.get("temperature_c", 98.6)
        spo2 = vitals.get("spo2", 99)
        heart_rate = vitals.get("heart_rate", 80)

        lab_results = {
            "cbc": {
                "wbc": "high" if "fever" in symptoms or temperature > 99.5 else "normal",
                "hemoglobin": "normal"
            },
            "crp": "elevated" if "fever" in symptoms else "normal",
            "blood_sugar": "normal"
        }

        scan_results = {
            "chest_xray": "mild infiltrates" if "cough" in symptoms and spo2 < 97 else "clear",
            "ecg": "mild sinus tachycardia" if heart_rate > 95 else "normal"
        }

        return {
            "ordered_tests": intake_output.get("required_steps", []),
            "lab_results": lab_results,
            "scan_results": scan_results
        }

    def run(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: Diagnostic Agent] Ordering tests and identifying possible conditions...")

        test_data = self.simulate_tests(patient_data, intake_output)

        system_prompt = (
            "You are Agent 2 in a hospital sequential multi-agent system. "
            "Your role is diagnostic interpretation only. "
            "Use the patient details and simulated tests to identify likely conditions. "
            "Return only valid JSON with exactly these keys: "
            "possible_conditions, diagnostic_reasoning, confidence_level, red_flags. "
            "Keep the response realistic, professional, and concise. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Simulated Test Data:
{json.dumps(test_data, indent=2)}

Output rules:
- possible_conditions: list of 2 to 4 likely conditions
- diagnostic_reasoning: 4 to 6 lines
- confidence_level: low, medium, medium-high, or high
- red_flags: short list
"""

        result = call_llm(system_prompt, user_prompt)
        diagnosis = safe_json_parse(result)
        diagnosis["test_data"] = test_data
        return diagnosis


class TreatmentAgent:
    def run(self, patient_data: Dict[str, Any], diagnosis_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Treatment Agent] Preparing treatment options based on diagnosis and preferences...")

        system_prompt = (
            "You are Agent 3 in a hospital sequential multi-agent system. "
            "Your role is treatment planning only. "
            "Use diagnosis, patient preferences, allergy information, and general safety principles. "
            "Return only valid JSON with exactly these keys: "
            "treatment_options, preferred_option, treatment_rationale, follow_up_advice. "
            "Keep treatment options broad, practical, and suitable for a hospital simulation. "
            "Avoid unnecessary over-detail. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Output rules:
- treatment_options: 2 to 4 options
- preferred_option: one selected option
- treatment_rationale: 4 to 6 lines
- follow_up_advice: concise and practical
- consider allergy and patient preference for home recovery if possible
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def run(
        self,
        patient_data: Dict[str, Any],
        intake_output: Dict[str, Any],
        diagnosis_output: Dict[str, Any],
        treatment_output: Dict[str, Any]
    ) -> Dict[str, Any]:
        print("\n[Agent 4: Decision Agent] Reviewing all inputs and generating final recommendation...")

        system_prompt = (
            "You are Agent 4 in a hospital sequential multi-agent system. "
            "Your role is final decision making. "
            "Review the intake, diagnostics, and treatment suggestions and deliver the final recommendation. "
            "Return only valid JSON with exactly these keys: "
            "final_recommendation, patient_friendly_explanation, escalation_needed, final_notes. "
            "Keep the answer professional, readable, and operationally useful. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Treatment Output:
{json.dumps(treatment_output, indent=2)}

Output rules:
- final_recommendation: 3 to 5 lines
- patient_friendly_explanation: simple language
- escalation_needed: true or false
- final_notes: concise internal note
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HospitalMultiAgentSystem:
    def __init__(self):
        self.intake_agent = IntakeAgent()
        self.diagnostic_agent = DiagnosticAgent()
        self.treatment_agent = TreatmentAgent()
        self.decision_agent = DecisionAgent()

    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("Patient Case Received:", patient_data.get("name", "Unknown Patient"))

        intake_output = self.intake_agent.run(patient_data)
        diagnosis_output = self.diagnostic_agent.run(patient_data, intake_output)
        treatment_output = self.treatment_agent.run(patient_data, diagnosis_output)
        decision_output = self.decision_agent.run(
            patient_data,
            intake_output,
            diagnosis_output,
            treatment_output
        )

        return {
            "patient_data": patient_data,
            "intake_output": intake_output,
            "diagnosis_output": diagnosis_output,
            "treatment_output": treatment_output,
            "decision_output": decision_output
        }


patient_case = {
    "name": "Rahul Sharma",
    "age": 45,
    "gender": "Male",
    "symptoms": ["fever", "cough", "fatigue"],
    "medical_history": ["hypertension"],
    "allergies": ["penicillin"],
    "vitals": {
        "temperature_c": 101.2,
        "blood_pressure": "140/90",
        "heart_rate": 96,
        "spo2": 95
    },
    "preferences": {
        "prefers_non_surgical": True,
        "prefers_home_recovery_if_possible": True
    }
}

system = HospitalMultiAgentSystem()
output = system.run(patient_case)

print("\n" + "=" * 70)
print("HOSPITAL SEQUENTIAL MULTI-AGENT SYSTEM OUTPUT")
print("=" * 70)

print("\n1. Intake Agent Output")
print(json.dumps(output["intake_output"], indent=2))

print("\n2. Diagnostic Agent Output")
print(json.dumps(output["diagnosis_output"], indent=2))

print("\n3. Treatment Agent Output")
print(json.dumps(output["treatment_output"], indent=2))

print("\n4. Decision Agent Output")
print(json.dumps(output["decision_output"], indent=2))

print("\n" + "=" * 70)
print("FINAL SYSTEM SUMMARY")
print("=" * 70)
print("This is a sequential multi-agent hospital workflow simulation.")
print("Agent 1 handled intake planning.")
print("Agent 2 handled diagnostics and test interpretation.")
print("Agent 3 handled treatment planning.")
print("Agent 4 handled the final decision.")
print("Each agent used the previous agent's output, which makes this a pipeline-style multi-agent system.")

In [8]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)
# ================================

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions
# ================================

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines
# ================================

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY ")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1200
    )

    content = completion.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class IntakeAgent:
    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 1: Intake Agent] Reviewing patient history and planning next steps...")

        system_prompt = (
            "You are Agent 1: Intake Agent in a hospital sequential multi-agent system. "
            "Your responsibility is only intake planning and first-level case structuring. "
            "You must review the patient profile carefully, including symptoms, age, gender, past medical history, allergies, vital signs, and preferences. "
            "Your objective is to create a strong intake-level understanding of the case and determine what the next immediate hospital workflow steps should be. "
            "You should think like a hospital intake coordinator who decides what needs to happen first before full diagnostics and treatment planning begin. "
            "Do not produce deep diagnosis, do not recommend final treatment, and do not make the final decision. "
            "Focus only on summary, required next steps, urgency, and operational intake notes. "
            "Return only valid JSON with exactly these keys: "
            "summary, required_steps, priority_level, intake_notes. "
            "Rules: "
            "summary must be concise, clinically relevant, and 2 to 4 lines long; "
            "required_steps must include only realistic immediate actions such as tests, monitoring, scans, consultations, or reviews; "
            "priority_level must be exactly one of low, medium, or high; "
            "intake_notes must provide practical notes for downstream agents in the pipeline. "
            "Strictly return only valid JSON. Do not include markdown, bullets outside JSON, explanations outside JSON, code fences, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as Agent 1: Intake Agent.

Patient Case:
{json.dumps(patient_data, indent=2)}

Your responsibilities:
1. Understand the patient’s presenting problem from the intake perspective.
2. Summarize the case in a practical and medically meaningful way.
3. Identify the immediate next workflow actions needed.
4. Respect allergies, medical history, and patient preferences.
5. Set an urgency level based on the intake view of the case.

Output guidance:
- summary should capture the patient’s condition and context clearly
- required_steps should contain short, realistic immediate actions
- priority_level should be low, medium, or high
- intake_notes should help the next agent continue the case efficiently

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DiagnosticAgent:
    def simulate_tests(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        symptoms = [s.lower() for s in patient_data.get("symptoms", [])]
        vitals = patient_data.get("vitals", {})
        temperature = vitals.get("temperature_c", 98.6)
        spo2 = vitals.get("spo2", 99)
        heart_rate = vitals.get("heart_rate", 80)

        lab_results = {
            "cbc": {
                "wbc": "high" if "fever" in symptoms or temperature > 99.5 else "normal",
                "hemoglobin": "normal"
            },
            "crp": "elevated" if "fever" in symptoms else "normal",
            "blood_sugar": "normal"
        }

        scan_results = {
            "chest_xray": "mild infiltrates" if "cough" in symptoms and spo2 < 97 else "clear",
            "ecg": "mild sinus tachycardia" if heart_rate > 95 else "normal"
        }

        return {
            "ordered_tests": intake_output.get("required_steps", []),
            "lab_results": lab_results,
            "scan_results": scan_results
        }

    def run(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: Diagnostic Agent] Ordering tests and identifying possible conditions...")

        test_data = self.simulate_tests(patient_data, intake_output)

        system_prompt = (
            "You are Agent 2: Diagnostic Agent in a hospital sequential multi-agent system. "
            "Your responsibility is only diagnostic interpretation. "
            "You receive patient details, the intake agent's structured output, and simulated test results. "
            "Your job is to identify the most likely possible medical conditions and explain the reasoning in a disciplined, evidence-based way. "
            "You must connect symptoms, medical history, vitals, and test findings logically. "
            "You are not the treatment planner and not the final decision maker. "
            "Do not produce final treatment plans. "
            "Return only valid JSON with exactly these keys: "
            "possible_conditions, diagnostic_reasoning, confidence_level, red_flags. "
            "Rules: "
            "possible_conditions should contain 2 to 4 clinically reasonable possibilities; "
            "diagnostic_reasoning should explain the interpretation in 4 to 7 lines, grounded in the provided data; "
            "confidence_level must be one of low, medium, medium-high, or high; "
            "red_flags must capture warning indicators, deterioration signs, or important clinical concerns. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, code fences, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as Agent 2: Diagnostic Agent.

Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Simulated Test Data:
{json.dumps(test_data, indent=2)}

Your responsibilities:
1. Interpret symptoms, history, vitals, and test results together.
2. Identify the most likely conditions or explanations.
3. Explain why these conditions are plausible using the evidence provided.
4. Mark any clinical warning signs that may need closer observation.
5. Keep the result suitable for the next treatment-planning stage.

Output guidance:
- possible_conditions should list 2 to 4 likely conditions
- diagnostic_reasoning should link evidence to the possible conditions
- confidence_level should reflect how strong the available evidence is
- red_flags should contain important warning points

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        diagnosis = safe_json_parse(result)
        diagnosis["test_data"] = test_data
        return diagnosis


class TreatmentAgent:
    def run(self, patient_data: Dict[str, Any], diagnosis_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Treatment Agent] Preparing treatment options based on diagnosis and preferences...")

        system_prompt = (
            "You are Agent 3: Treatment Agent in a hospital sequential multi-agent system. "
            "Your responsibility is only treatment planning. "
            "You receive the patient profile and the diagnostic findings. "
            "Your job is to propose practical treatment options that align with the likely diagnosis, patient allergies, patient preferences, and basic safety principles. "
            "You must think like a treatment planner preparing options for review, not like the final authority. "
            "Do not act as the final decision maker. "
            "Do not produce unnecessarily extreme or overly detailed responses. "
            "Keep the treatment planning realistic for a hospital workflow simulation. "
            "Return only valid JSON with exactly these keys: "
            "treatment_options, preferred_option, treatment_rationale, follow_up_advice. "
            "Rules: "
            "treatment_options should contain 2 to 4 realistic care options; "
            "preferred_option should identify the best current option based on the available case information; "
            "treatment_rationale should explain the selection in 4 to 7 lines, considering symptoms, diagnosis, allergy information, patient preference, and safety; "
            "follow_up_advice should contain short and practical next-step guidance such as monitoring, review timing, and escalation advice. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, code fences, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as Agent 3: Treatment Agent.

Patient Data:
{json.dumps(patient_data, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Your responsibilities:
1. Suggest realistic treatment options based on the likely diagnosis.
2. Consider the penicillin allergy carefully.
3. Respect the patient's preference for non-surgical and home-based recovery when clinically reasonable.
4. Provide a preferred option and explain why it is the best fit at this stage.
5. Provide concise follow-up and monitoring advice for the next step in care.

Output guidance:
- treatment_options should contain 2 to 4 realistic options
- preferred_option should name the best current option
- treatment_rationale should explain the choice clearly
- follow_up_advice should be concise, practical, and clinically sensible

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def run(
        self,
        patient_data: Dict[str, Any],
        intake_output: Dict[str, Any],
        diagnosis_output: Dict[str, Any],
        treatment_output: Dict[str, Any]
    ) -> Dict[str, Any]:
        print("\n[Agent 4: Decision Agent] Reviewing all inputs and generating final recommendation...")

        system_prompt = (
            "You are Agent 4: Decision Agent in a hospital sequential multi-agent system. "
            "Your responsibility is final decision synthesis. "
            "You receive the original patient case, the intake summary, the diagnostic interpretation, and the treatment planning output. "
            "Your job is to review all prior agent outputs and generate the final recommendation for this hospital workflow simulation. "
            "You must combine all earlier reasoning into one clear, practical, professional final answer. "
            "Your response should be balanced, realistic, and operationally useful. "
            "Return only valid JSON with exactly these keys: "
            "final_recommendation, patient_friendly_explanation, escalation_needed, final_notes. "
            "Rules: "
            "final_recommendation should be 3 to 5 lines and summarize the best immediate path forward; "
            "patient_friendly_explanation should explain the situation in simpler language suitable for a patient; "
            "escalation_needed must be true or false only; "
            "final_notes should provide a short internal note for workflow continuity. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, code fences, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as Agent 4: Decision Agent.

Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Treatment Output:
{json.dumps(treatment_output, indent=2)}

Your responsibilities:
1. Review all previous agent outputs together.
2. Produce the final recommendation for the case.
3. Make the recommendation practical and easy to understand.
4. Explain the recommendation both professionally and in patient-friendly language.
5. Indicate whether escalation is currently needed.

Output guidance:
- final_recommendation should summarize the immediate path forward
- patient_friendly_explanation should be simple and understandable
- escalation_needed should be true or false
- final_notes should support internal continuity in the workflow

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HospitalMultiAgentSystem:
    def __init__(self):
        self.intake_agent = IntakeAgent()
        self.diagnostic_agent = DiagnosticAgent()
        self.treatment_agent = TreatmentAgent()
        self.decision_agent = DecisionAgent()

    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("Patient Case Received:", patient_data.get("name", "Unknown Patient"))

        intake_output = self.intake_agent.run(patient_data)
        diagnosis_output = self.diagnostic_agent.run(patient_data, intake_output)
        treatment_output = self.treatment_agent.run(patient_data, diagnosis_output)
        decision_output = self.decision_agent.run(
            patient_data,
            intake_output,
            diagnosis_output,
            treatment_output
        )

        return {
            "patient_data": patient_data,
            "intake_output": intake_output,
            "diagnosis_output": diagnosis_output,
            "treatment_output": treatment_output,
            "decision_output": decision_output
        }


patient_case = {
    "name": "Rahul Sharma",
    "age": 45,
    "gender": "Male",
    "symptoms": ["fever", "cough", "fatigue"],
    "medical_history": ["hypertension"],
    "allergies": ["penicillin"],
    "vitals": {
        "temperature_c": 101.2,
        "blood_pressure": "140/90",
        "heart_rate": 96,
        "spo2": 95
    },
    "preferences": {
        "prefers_non_surgical": True,
        "prefers_home_recovery_if_possible": True
    }
}

system = HospitalMultiAgentSystem()
output = system.run(patient_case)

print("\n" + "=" * 70)
print("HOSPITAL SEQUENTIAL MULTI-AGENT SYSTEM OUTPUT")
print("=" * 70)

print("\n1. Intake Agent Output")
print(json.dumps(output["intake_output"], indent=2))

print("\n2. Diagnostic Agent Output")
print(json.dumps(output["diagnosis_output"], indent=2))

print("\n3. Treatment Agent Output")
print(json.dumps(output["treatment_output"], indent=2))

print("\n4. Decision Agent Output")
print(json.dumps(output["decision_output"], indent=2))

print("\n" + "=" * 70)
print("FINAL SYSTEM SUMMARY")
print("=" * 70)
print("This is a sequential multi-agent hospital workflow simulation.")
print("Agent 1 handled intake planning.")
print("Agent 2 handled diagnostics and test interpretation.")
print("Agent 3 handled treatment planning.")
print("Agent 4 handled the final decision.")
print("Each agent used the previous agent's output, which makes this a pipeline-style multi-agent system.")

In [11]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market.
# The Manager Agent oversees the process and delegates tasks to specialized workers.

# ================================
# MANAGER AGENT
# - Receives the overall goal
# - Dynamically assigns tasks to worker agents depending on what is needed
# ================================

# ================================
# WORKER AGENTS
# - Market Research Worker
# - Finance Worker
# - Operations Worker
# - Legal Worker
# - HR Worker
# ================================

import json


class MarketResearchWorker:
    def run(self, product_data):
        print("\n[Market Research Worker] Collecting market trends, customer demand, and competitor data...")
        return {
            "worker": "Market Research Worker",
            "report": {
                "demand_forecast": "High demand in Tier-1 cities and growing interest in Tier-2 cities",
                "competition_level": "Moderate competition with 3 major competitors",
                "customer_preferences": [
                    "Affordable pricing",
                    "Strong after-sales support",
                    "Localized features"
                ],
                "market_insight": "The product has strong potential if positioned with competitive pricing and local customization."
            }
        }


class FinanceWorker:
    def run(self, product_data):
        print("\n[Finance Worker] Evaluating investment, ROI, and pricing strategy...")
        return {
            "worker": "Finance Worker",
            "report": {
                "estimated_investment": "$10M",
                "expected_roi_period": "2 years",
                "pricing_strategy": "Penetration pricing in launch phase, then value-based pricing",
                "financial_risk": "Medium",
                "financial_insight": "The launch is financially feasible if customer acquisition costs remain controlled."
            }
        }


class OperationsWorker:
    def run(self, product_data):
        print("\n[Operations Worker] Reviewing supply chain, production, and logistics...")
        return {
            "worker": "Operations Worker",
            "report": {
                "production_capacity": "Factories can scale up production by 35%",
                "supply_chain_status": "Stable suppliers available in the region",
                "logistics_challenge": "Shipping costs are higher in remote regions",
                "operations_insight": "Operational readiness is good, but logistics optimization is required."
            }
        }


class LegalWorker:
    def run(self, product_data):
        print("\n[Legal Worker] Checking compliance, certifications, and intellectual property...")
        return {
            "worker": "Legal Worker",
            "report": {
                "trademark_status": "Trademark available in target markets",
                "compliance_status": "Import regulations require certification before launch",
                "legal_risk": "Medium",
                "legal_insight": "The launch is possible, but certification and regulatory clearance must be completed first."
            }
        }


class HRWorker:
    def run(self, product_data):
        print("\n[HR Worker] Assessing hiring and training needs...")
        return {
            "worker": "HR Worker",
            "report": {
                "new_hires_required": 50,
                "roles_needed": [
                    "Customer Support Executives",
                    "Sales Associates",
                    "Regional Managers"
                ],
                "training_need": "Product training and regional sales onboarding required",
                "hr_insight": "Workforce expansion is manageable with phased hiring."
            }
        }


class ManagerAgent:
    def __init__(self):
        self.market_worker = MarketResearchWorker()
        self.finance_worker = FinanceWorker()
        self.operations_worker = OperationsWorker()
        self.legal_worker = LegalWorker()
        self.hr_worker = HRWorker()

    def create_plan(self, goal, product_data):
        print("[Manager Agent] Analyzing business goal and dynamically assigning tasks...")

        plan = []

        if "market" in goal.lower() or "launch" in goal.lower() or "feasibility" in goal.lower():
            plan.append("market_research")
            plan.append("finance")
            plan.append("operations")
            plan.append("legal")
            plan.append("hr")

        if product_data.get("budget_unclear", False) and "finance" not in plan:
            plan.append("finance")

        if product_data.get("regulations_complex", False) and "legal" not in plan:
            plan.append("legal")

        return plan

    def final_decision(self, results):
        print("\n[Manager Agent] Consolidating reports and generating final strategy recommendation...")

        decision = {
            "launch_feasibility": "Feasible with conditions",
            "key_strengths": [
                "Strong demand in urban markets",
                "Manageable competition",
                "Good operational scalability"
            ],
            "key_risks": [
                "Regulatory certification delays",
                "High logistics cost in remote areas",
                "Need for careful budget execution"
            ],
            "final_recommendation": (
                "Launch Product Y in Asia through a phased strategy starting with Tier-1 cities. "
                "Complete legal certifications first, use competitive entry pricing, "
                "optimize logistics partnerships, and begin phased hiring before full rollout."
            )
        }

        return decision

    def run(self, goal, product_data):
        print("Goal:", goal)

        plan = self.create_plan(goal, product_data)

        results = {}

        for step in plan:
            if step == "market_research":
                results["market_research"] = self.market_worker.run(product_data)

            elif step == "finance":
                results["finance"] = self.finance_worker.run(product_data)

            elif step == "operations":
                results["operations"] = self.operations_worker.run(product_data)

            elif step == "legal":
                results["legal"] = self.legal_worker.run(product_data)

            elif step == "hr":
                results["hr"] = self.hr_worker.run(product_data)

        results["manager_decision"] = self.final_decision(results)
        return results


# ================================
# INPUT DATA
# ================================
goal = "Evaluate feasibility of launching Product Y in Asia"

product_data = {
    "product_name": "Product Y",
    "target_region": "Asia",
    "budget_unclear": True,
    "regulations_complex": True,
    "production_ready": True,
    "staffing_expansion_needed": True
}

# ================================
# RUN SYSTEM
# ================================
manager = ManagerAgent()
final_output = manager.run(goal, product_data)

print("\n" + "=" * 70)
print("CORPORATE MARKET RESEARCH & STRATEGY OUTPUT")
print("=" * 70)

print("\n1. Market Research Report")
print(json.dumps(final_output["market_research"], indent=2))

print("\n2. Finance Report")
print(json.dumps(final_output["finance"], indent=2))

print("\n3. Operations Report")
print(json.dumps(final_output["operations"], indent=2))

print("\n4. Legal Report")
print(json.dumps(final_output["legal"], indent=2))

print("\n5. HR Report")
print(json.dumps(final_output["hr"], indent=2))

print("\n6. Manager Final Decision")
print(json.dumps(final_output["manager_decision"], indent=2))

Goal: Evaluate feasibility of launching Product Y in Asia
[Manager Agent] Analyzing business goal and dynamically assigning tasks...

[Market Research Worker] Collecting market trends, customer demand, and competitor data...

[Finance Worker] Evaluating investment, ROI, and pricing strategy...

[Operations Worker] Reviewing supply chain, production, and logistics...

[Legal Worker] Checking compliance, certifications, and intellectual property...

[HR Worker] Assessing hiring and training needs...

[Manager Agent] Consolidating reports and generating final strategy recommendation...

CORPORATE MARKET RESEARCH & STRATEGY OUTPUT

1. Market Research Report
{
  "worker": "Market Research Worker",
  "report": {
    "demand_forecast": "High demand in Tier-1 cities and growing interest in Tier-2 cities",
    "competition_level": "Moderate competition with 3 major competitors",
    "customer_preferences": [
      "Affordable pricing",
      "Strong after-sales support",
      "Localized featur

In [12]:
# GROQ API key removed for security
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market.
# The Manager Agent oversees the process and delegates tasks to specialized workers.

# ================================
# MANAGER AGENT
# - Receives the overall goal
# - Dynamically assigns tasks to worker agents depending on business needs
# - Consolidates all worker reports into a final recommendation
# ================================

# ================================
# WORKER AGENTS
# - Market Research Worker
# - Finance Worker
# - Operations Worker
# - Legal Worker
# - HR Worker
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class ManagerAgent:
    def create_plan(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Manager Agent] Reviewing strategic goal and assigning worker tasks...")

        system_prompt = (
            "You are the Manager Agent in a dynamic corporate multi-agent strategy system. "
            "Your role is to analyze the business objective and decide which specialized worker agents should be activated. "
            "You are responsible for strategic task orchestration only. "
            "Available worker agents are: Market Research Worker, Finance Worker, Operations Worker, Legal Worker, and HR Worker. "
            "You must examine the goal and the product context carefully, then create a practical task execution plan. "
            "If market fit is unclear, include Market Research Worker. "
            "If investment, pricing, ROI, or budget is important, include Finance Worker. "
            "If supply chain, manufacturing, or logistics is important, include Operations Worker. "
            "If regulations, certifications, compliance, import rules, contracts, or IP are important, include Legal Worker. "
            "If scaling teams, staffing, hiring, or training is relevant, include HR Worker. "
            "Return only valid JSON with exactly these keys: "
            "goal_summary, assigned_workers, planning_rationale, manager_notes. "
            "Rules: "
            "goal_summary should summarize the strategic objective in 2 to 4 lines; "
            "assigned_workers should be a list containing worker names exactly as defined; "
            "planning_rationale should explain why these workers are needed in 4 to 6 lines; "
            "manager_notes should be concise operational guidance for execution. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following business case as the Manager Agent.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Understand the strategic objective clearly.
2. Identify which worker agents are required.
3. Build a practical execution plan.
4. Ensure the plan covers risk, feasibility, operations, compliance, and execution readiness where relevant.

Output guidance:
- goal_summary should describe the strategic objective clearly
- assigned_workers should include only relevant worker agents
- planning_rationale should justify the delegation strategy
- manager_notes should guide execution flow

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)

    def final_decision(self, goal: str, product_data: Dict[str, Any], worker_reports: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Manager Agent] Consolidating all worker reports into final business recommendation...")

        system_prompt = (
            "You are the Manager Agent in a corporate multi-agent business strategy system. "
            "Your responsibility now is final strategic synthesis. "
            "You have already received detailed reports from specialized worker agents. "
            "Your task is to review all worker outputs together and provide the final business recommendation regarding the product launch. "
            "You must combine strategic, financial, operational, legal, and workforce considerations into a single decision. "
            "Keep the response executive-friendly, clear, and practically useful for leadership review. "
            "Return only valid JSON with exactly these keys: "
            "launch_feasibility, key_strengths, key_risks, final_recommendation, execution_strategy. "
            "Rules: "
            "launch_feasibility should be one of Not Feasible, Feasible with Conditions, or Highly Feasible; "
            "key_strengths should be a short list of the strongest positive signals; "
            "key_risks should be a short list of the major constraints or risks; "
            "final_recommendation should be 4 to 7 lines and clearly state whether and how the launch should proceed; "
            "execution_strategy should explain a recommended rollout approach in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the Manager Agent for final decision making.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Worker Reports:
{json.dumps(worker_reports, indent=2)}

Your responsibilities:
1. Review all worker reports together.
2. Assess the overall feasibility of launching the product.
3. Identify the strongest reasons to proceed.
4. Identify the most important risks or blockers.
5. Provide a practical final recommendation and rollout strategy.

Output guidance:
- launch_feasibility should reflect the overall business outlook
- key_strengths should highlight the strongest supporting factors
- key_risks should capture the major concerns
- final_recommendation should clearly advise the leadership team
- execution_strategy should describe how the company should move forward

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class MarketResearchWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Market Research Worker] Collecting competitor data, customer preferences, and demand forecasts...")

        system_prompt = (
            "You are the Market Research Worker in a corporate multi-agent strategy system. "
            "Your responsibility is market intelligence only. "
            "You must analyze customer demand, competitor intensity, buyer preferences, and regional opportunity signals. "
            "Think like a market analyst preparing a focused report for the strategy manager. "
            "Do not discuss internal company staffing, legal compliance, or operational execution in depth. "
            "Return only valid JSON with exactly these keys: "
            "demand_forecast, competition_level, customer_preferences, market_insight. "
            "Rules: "
            "demand_forecast should summarize expected market demand clearly; "
            "competition_level should describe the competitive intensity; "
            "customer_preferences should be a short list of major customer expectations; "
            "market_insight should provide a concise strategic interpretation in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the Market Research Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Assess likely customer demand.
2. Estimate competition level.
3. Identify important customer preferences.
4. Provide a practical market insight for leadership.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinanceWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Finance Worker] Evaluating budget, ROI, and pricing strategy...")

        system_prompt = (
            "You are the Finance Worker in a corporate multi-agent strategy system. "
            "Your responsibility is financial feasibility only. "
            "You must evaluate investment needs, expected ROI, pricing approach, and financial risk. "
            "Think like a strategy finance analyst preparing a concise board-style note. "
            "Do not focus on legal or staffing matters. "
            "Return only valid JSON with exactly these keys: "
            "estimated_investment, expected_roi_period, pricing_strategy, financial_risk, financial_insight. "
            "Rules: "
            "estimated_investment should provide a realistic high-level investment estimate; "
            "expected_roi_period should estimate business recovery or return period; "
            "pricing_strategy should define a sensible market entry pricing approach; "
            "financial_risk should be low, medium, or high; "
            "financial_insight should summarize financial feasibility in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the Finance Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Estimate investment scale.
2. Estimate likely ROI timeline.
3. Suggest an appropriate pricing strategy.
4. Identify the overall financial risk.
5. Provide a practical finance insight for management.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class OperationsWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Operations Worker] Reviewing supply chain, production capacity, and logistics...")

        system_prompt = (
            "You are the Operations Worker in a corporate multi-agent strategy system. "
            "Your responsibility is operational feasibility only. "
            "You must evaluate production readiness, supply chain stability, logistics practicality, and scale-up capability. "
            "Think like an operations strategy analyst supporting a product launch review. "
            "Return only valid JSON with exactly these keys: "
            "production_capacity, supply_chain_status, logistics_challenge, operations_insight. "
            "Rules: "
            "production_capacity should describe current scale-up readiness; "
            "supply_chain_status should evaluate supplier and execution stability; "
            "logistics_challenge should identify the main operational difficulty; "
            "operations_insight should summarize whether operations can support launch in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the Operations Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Assess whether operations can support launch.
2. Review supply chain reliability.
3. Review production readiness.
4. Identify the biggest logistics concern.
5. Provide a concise operational insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Legal Worker] Reviewing compliance, intellectual property, and regulations...")

        system_prompt = (
            "You are the Legal Worker in a corporate multi-agent strategy system. "
            "Your responsibility is legal and compliance feasibility only. "
            "You must review the risk of regulatory complexity, certification requirements, compliance barriers, and intellectual property availability. "
            "Think like a legal strategy reviewer preparing a practical launch-readiness note. "
            "Return only valid JSON with exactly these keys: "
            "trademark_status, compliance_status, legal_risk, legal_insight. "
            "Rules: "
            "trademark_status should indicate whether brand/IP availability looks favorable; "
            "compliance_status should summarize regulatory or certification requirements; "
            "legal_risk should be low, medium, or high; "
            "legal_insight should explain the legal feasibility in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the Legal Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Assess trademark or IP readiness.
2. Review likely compliance obligations.
3. Evaluate legal risk.
4. Highlight the biggest legal barrier, if any.
5. Provide a concise legal insight for leadership.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HRWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[HR Worker] Assessing staffing needs and training requirements...")

        system_prompt = (
            "You are the HR Worker in a corporate multi-agent strategy system. "
            "Your responsibility is workforce readiness only. "
            "You must evaluate staffing requirements, role needs, hiring scale, and training readiness for launch execution. "
            "Think like an HR planning lead supporting commercial expansion. "
            "Return only valid JSON with exactly these keys: "
            "new_hires_required, roles_needed, training_need, hr_insight. "
            "Rules: "
            "new_hires_required should estimate the required hiring scale; "
            "roles_needed should be a short list of the most important roles; "
            "training_need should summarize the type of onboarding or capability building required; "
            "hr_insight should explain workforce readiness in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following case as the HR Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Your responsibilities:
1. Estimate hiring needs.
2. Identify which roles are most important for launch.
3. Assess what training will be needed.
4. Provide an HR readiness insight for management.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CorporateStrategySystem:
    def __init__(self):
        self.manager = ManagerAgent()
        self.market_worker = MarketResearchWorker()
        self.finance_worker = FinanceWorker()
        self.operations_worker = OperationsWorker()
        self.legal_worker = LegalWorker()
        self.hr_worker = HRWorker()

    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        manager_plan = self.manager.create_plan(goal, product_data)

        assigned_workers = manager_plan.get("assigned_workers", [])
        worker_reports = {}

        for worker_name in assigned_workers:
            if worker_name == "Market Research Worker":
                worker_reports["market_research_report"] = self.market_worker.run(goal, product_data)

            elif worker_name == "Finance Worker":
                worker_reports["finance_report"] = self.finance_worker.run(goal, product_data)

            elif worker_name == "Operations Worker":
                worker_reports["operations_report"] = self.operations_worker.run(goal, product_data)

            elif worker_name == "Legal Worker":
                worker_reports["legal_report"] = self.legal_worker.run(goal, product_data)

            elif worker_name == "HR Worker":
                worker_reports["hr_report"] = self.hr_worker.run(goal, product_data)

        final_decision = self.manager.final_decision(goal, product_data, worker_reports)

        return {
            "manager_plan": manager_plan,
            "worker_reports": worker_reports,
            "manager_final_decision": final_decision
        }


goal = "Evaluate feasibility of launching Product Y in Asia"

product_data = {
    "product_name": "Product Y",
    "target_region": "Asia",
    "industry": "Consumer Technology",
    "budget_unclear": True,
    "regulations_complex": True,
    "production_ready": True,
    "staffing_expansion_needed": True,
    "target_customers": "Urban and digitally active middle-income consumers",
    "launch_objective": "Establish strong presence in high-demand Asian metro markets first"
}

system = CorporateStrategySystem()
final_output = system.run(goal, product_data)

print("\n" + "=" * 70)
print("CORPORATE MARKET RESEARCH & STRATEGY OUTPUT")
print("=" * 70)

print("\n1. Manager Planning Output")
print(json.dumps(final_output["manager_plan"], indent=2))

if "market_research_report" in final_output["worker_reports"]:
    print("\n2. Market Research Worker Output")
    print(json.dumps(final_output["worker_reports"]["market_research_report"], indent=2))

if "finance_report" in final_output["worker_reports"]:
    print("\n3. Finance Worker Output")
    print(json.dumps(final_output["worker_reports"]["finance_report"], indent=2))

if "operations_report" in final_output["worker_reports"]:
    print("\n4. Operations Worker Output")
    print(json.dumps(final_output["worker_reports"]["operations_report"], indent=2))

if "legal_report" in final_output["worker_reports"]:
    print("\n5. Legal Worker Output")
    print(json.dumps(final_output["worker_reports"]["legal_report"], indent=2))

if "hr_report" in final_output["worker_reports"]:
    print("\n6. HR Worker Output")
    print(json.dumps(final_output["worker_reports"]["hr_report"], indent=2))

print("\n7. Manager Final Decision")
print(json.dumps(final_output["manager_final_decision"], indent=2))

print("\n" + "=" * 70)
print("FINAL SYSTEM SUMMARY")
print("=" * 70)
print("This is a dynamic multi-agent corporate strategy system.")
print("The Manager Agent analyzed the strategic goal and assigned specialized worker agents.")
print("Each worker produced a focused report in its own domain.")
print("The Manager Agent then consolidated all worker outputs into a final launch recommendation.")
print("This demonstrates manager-worker orchestration in a multiple-agent business strategy workflow.")

In [13]:
# Scenario: Crisis Response Broadcast Multi-Agent System
# A company detects a data breach in its customer database.
# One broadcasting agent alerts all specialized agents simultaneously.
# Each agent responds from its own domain.
# A final coordinator agent combines all responses into one crisis response plan.

# ================================
# AGENT 1: Crisis Coordinator (Broadcaster)
# - Broadcasts the incident to all other agents at the same time
# ================================

# ================================
# AGENT 2: IT Security Agent
# - Handles containment, patching, and forensic response
# ================================

# ================================
# AGENT 3: Communications Agent
# - Handles internal and external communication strategy
# ================================

# ================================
# AGENT 4: Finance Agent
# - Evaluates financial impact and emergency budget actions
# ================================

# ================================
# AGENT 5: Legal Agent
# - Reviews compliance, liability, and regulatory obligations
# ================================

# ================================
# AGENT 6: HR Agent
# - Supports employee briefing and internal coordination
# ================================

# ================================
# AGENT 7: Decision Agent (Coordinator)
# - Collects all responses
# - Produces the final integrated crisis response plan
# ================================

import json


class CrisisCoordinatorAgent:
    def broadcast(self, incident_message):
        print("[Agent 1: Crisis Coordinator] Broadcasting incident to all agents...")
        return {
            "incident_type": "Data Breach",
            "broadcast_message": incident_message,
            "priority": "Critical",
            "status": "Broadcast delivered to all response agents"
        }


class ITSecurityAgent:
    def respond(self, broadcast_data):
        print("\n[Agent 2: IT Security Agent] Responding to security incident...")
        return {
            "agent": "IT Security Agent",
            "response": {
                "immediate_actions": [
                    "Isolate affected servers",
                    "Patch identified vulnerabilities",
                    "Start forensic analysis"
                ],
                "technical_priority": "Highest",
                "security_note": "Containment and root cause investigation must begin immediately to prevent further data exposure."
            }
        }


class CommunicationsAgent:
    def respond(self, broadcast_data):
        print("\n[Agent 3: Communications Agent] Preparing communication response...")
        return {
            "agent": "Communications Agent",
            "response": {
                "immediate_actions": [
                    "Draft internal memo",
                    "Prepare press release",
                    "Notify key stakeholders"
                ],
                "communication_priority": "High",
                "communication_note": "Clear and timely messaging is necessary to control misinformation and maintain trust."
            }
        }


class FinanceAgent:
    def respond(self, broadcast_data):
        print("\n[Agent 4: Finance Agent] Assessing financial exposure...")
        return {
            "agent": "Finance Agent",
            "response": {
                "immediate_actions": [
                    "Estimate financial impact",
                    "Allocate emergency funds",
                    "Review insurance coverage"
                ],
                "financial_priority": "High",
                "finance_note": "Rapid financial assessment is required to support incident containment and recovery planning."
            }
        }


class LegalAgent:
    def respond(self, broadcast_data):
        print("\n[Agent 5: Legal Agent] Reviewing regulatory and liability requirements...")
        return {
            "agent": "Legal Agent",
            "response": {
                "immediate_actions": [
                    "Assess regulatory obligations",
                    "Prepare compliance reports",
                    "Advise on liability exposure"
                ],
                "legal_priority": "High",
                "legal_note": "Compliance deadlines and legal exposure must be reviewed immediately to reduce regulatory risk."
            }
        }


class HRAgent:
    def respond(self, broadcast_data):
        print("\n[Agent 6: HR Agent] Supporting employee readiness and internal guidance...")
        return {
            "agent": "HR Agent",
            "response": {
                "immediate_actions": [
                    "Brief employees",
                    "Provide guidance for handling customer queries",
                    "Ensure morale support"
                ],
                "hr_priority": "Medium",
                "hr_note": "Employees must receive clear instructions to avoid panic, confusion, or inconsistent customer interaction."
            }
        }


class DecisionAgent:
    def integrate(self, broadcast_data, agent_responses):
        print("\n[Agent 7: Decision Agent] Integrating all responses into final crisis response plan...")

        final_plan = {
            "incident_summary": broadcast_data["broadcast_message"],
            "integrated_response_plan": (
                "Servers isolated, vulnerabilities being patched, forensic analysis initiated, "
                "internal and external communications prepared, financial impact under review, "
                "compliance obligations being addressed, and employees briefed with handling guidance."
            ),
            "priority_actions": [
                "Contain the breach technically",
                "Control communication flow",
                "Assess financial damage",
                "Ensure legal and regulatory compliance",
                "Stabilize employee response"
            ],
            "final_status": "Coordinated crisis response activated successfully"
        }

        return {
            "broadcast_data": broadcast_data,
            "all_agent_responses": agent_responses,
            "final_crisis_plan": final_plan
        }


incident_message = "Data breach detected in customer database. Immediate response required."

agent1 = CrisisCoordinatorAgent()
agent2 = ITSecurityAgent()
agent3 = CommunicationsAgent()
agent4 = FinanceAgent()
agent5 = LegalAgent()
agent6 = HRAgent()
agent7 = DecisionAgent()

broadcast_data = agent1.broadcast(incident_message)

responses = {
    "it_security": agent2.respond(broadcast_data),
    "communications": agent3.respond(broadcast_data),
    "finance": agent4.respond(broadcast_data),
    "legal": agent5.respond(broadcast_data),
    "hr": agent6.respond(broadcast_data)
}

final_output = agent7.integrate(broadcast_data, responses)

print("\n" + "=" * 72)
print("CRISIS RESPONSE BROADCAST MULTI-AGENT SYSTEM OUTPUT")
print("=" * 72)

print("\n1. Broadcast Output")
print(json.dumps(final_output["broadcast_data"], indent=2))

print("\n2. IT Security Agent Output")
print(json.dumps(final_output["all_agent_responses"]["it_security"], indent=2))

print("\n3. Communications Agent Output")
print(json.dumps(final_output["all_agent_responses"]["communications"], indent=2))

print("\n4. Finance Agent Output")
print(json.dumps(final_output["all_agent_responses"]["finance"], indent=2))

print("\n5. Legal Agent Output")
print(json.dumps(final_output["all_agent_responses"]["legal"], indent=2))

print("\n6. HR Agent Output")
print(json.dumps(final_output["all_agent_responses"]["hr"], indent=2))

print("\n7. Final Crisis Response Plan")
print(json.dumps(final_output["final_crisis_plan"], indent=2))

print("\n" + "=" * 72)
print("FINAL SYSTEM SUMMARY")
print("=" * 72)
print("This is a broadcast-style multi-agent crisis response system.")
print("Agent 1 broadcasted the incident to all specialized agents simultaneously.")
print("Each specialized agent produced a domain-specific response.")
print("Agent 7 integrated all responses into one final coordinated crisis response plan.")
print("This demonstrates parallel-style multi-agent coordination through broadcast communication.")

[Agent 1: Crisis Coordinator] Broadcasting incident to all agents...

[Agent 2: IT Security Agent] Responding to security incident...

[Agent 3: Communications Agent] Preparing communication response...

[Agent 4: Finance Agent] Assessing financial exposure...

[Agent 5: Legal Agent] Reviewing regulatory and liability requirements...

[Agent 6: HR Agent] Supporting employee readiness and internal guidance...

[Agent 7: Decision Agent] Integrating all responses into final crisis response plan...

CRISIS RESPONSE BROADCAST MULTI-AGENT SYSTEM OUTPUT

1. Broadcast Output
{
  "incident_type": "Data Breach",
  "broadcast_message": "Data breach detected in customer database. Immediate response required.",
  "priority": "Critical",
  "status": "Broadcast delivered to all response agents"
}

2. IT Security Agent Output
{
  "agent": "IT Security Agent",
  "response": {
    "immediate_actions": [
      "Isolate affected servers",
      "Patch identified vulnerabilities",
      "Start forensic ana

In [16]:
# GROQ API key removed for security
# with api

# Scenario: Crisis Response Broadcast Multi-Agent System
# A company detects a serious data breach in its customer database.
# One coordinator agent broadcasts the crisis message to all specialist agents simultaneously.
# Each specialist agent responds with domain-specific actions.
# A final decision agent integrates all responses into one unified crisis response plan.

# ================================
# AGENT 1: Crisis Coordinator (Broadcaster)
# - Broadcasts the crisis message to all agents simultaneously
# ================================

# ================================
# AGENT 2: IT Security Agent
# - Handles containment, patching, server isolation, and forensic analysis
# ================================

# ================================
# AGENT 3: Communications Agent
# - Handles internal memo, stakeholder messaging, and public communication
# ================================

# ================================
# AGENT 4: Finance Agent
# - Assesses business impact, emergency budget, and insurance considerations
# ================================

# ================================
# AGENT 5: Legal Agent
# - Handles compliance obligations, liability, and regulatory reporting
# ================================

# ================================
# AGENT 6: HR Agent
# - Handles employee guidance, morale, and internal support communication
# ================================

# ================================
# AGENT 7: Decision Agent (Coordinator)
# - Collects all responses
# - Produces the final integrated crisis response plan
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class CrisisCoordinatorAgent:
    def broadcast(self, incident_data: Dict[str, Any]) -> Dict[str, Any]:
        print("[Agent 1: Crisis Coordinator] Broadcasting incident to all specialist agents...")

        system_prompt = (
            "You are Agent 1: Crisis Coordinator in a broadcast-style enterprise incident response multi-agent system. "
            "Your responsibility is to convert the raw incident into a clear crisis broadcast for all specialist agents. "
            "You must act like a central crisis command function. "
            "Your broadcast should clearly summarize the incident, assign urgency, identify business-critical impact areas, "
            "and prepare all downstream specialist agents for immediate parallel response. "
            "Return only valid JSON with exactly these keys: "
            "incident_summary, broadcast_message, priority_level, impacted_areas, coordinator_note. "
            "Rules: "
            "incident_summary should be a concise summary of the event in 2 to 4 lines; "
            "broadcast_message should be a direct action-oriented crisis broadcast statement; "
            "priority_level must be one of High, Critical, or Severe; "
            "impacted_areas should be a short list of likely affected business areas; "
            "coordinator_note should provide a concise internal coordination note. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following incident as Agent 1: Crisis Coordinator.

Incident Data:
{json.dumps(incident_data, indent=2)}

Your responsibilities:
1. Understand the incident clearly.
2. Convert it into an enterprise-wide crisis broadcast.
3. Identify the likely affected areas.
4. Set the urgency level.
5. Prepare downstream specialist agents for rapid response.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class ITSecurityAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: IT Security Agent] Responding to the broadcasted cyber incident...")

        system_prompt = (
            "You are Agent 2: IT Security Agent in a crisis response multi-agent system. "
            "Your responsibility is technical containment and cybersecurity response only. "
            "You receive a crisis broadcast regarding a data breach incident. "
            "Your job is to provide the most relevant technical response actions, containment priorities, and security assessment notes. "
            "Think like a senior incident response lead. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, technical_priority, security_risk_level, security_note. "
            "Rules: "
            "immediate_actions should be a short list of the most important technical steps; "
            "technical_priority should describe urgency in plain language; "
            "security_risk_level must be Low, Medium, High, or Critical; "
            "security_note should explain the technical response logic in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following broadcast as Agent 2: IT Security Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Your responsibilities:
1. Suggest the first technical containment actions.
2. Identify technical risk severity.
3. Recommend technical incident response priorities.
4. Provide a concise security response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CommunicationsAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Communications Agent] Preparing internal and external communication actions...")

        system_prompt = (
            "You are Agent 3: Communications Agent in a crisis response multi-agent system. "
            "Your responsibility is communication strategy only. "
            "You receive a crisis broadcast about a data breach. "
            "Your task is to define how the company should handle internal messaging, stakeholder communication, and public communication readiness. "
            "Think like a crisis communications lead. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, communication_priority, audience_groups, communication_note. "
            "Rules: "
            "immediate_actions should be a short list of communication actions; "
            "communication_priority should reflect urgency in plain language; "
            "audience_groups should identify the most relevant communication targets; "
            "communication_note should explain the communication response logic in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following broadcast as Agent 3: Communications Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Your responsibilities:
1. Define immediate communication actions.
2. Identify who must be informed first.
3. Prepare the communication approach for internal and external messaging.
4. Provide a concise communication strategy note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinanceAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 4: Finance Agent] Assessing emergency financial exposure and support requirements...")

        system_prompt = (
            "You are Agent 4: Finance Agent in a crisis response multi-agent system. "
            "Your responsibility is financial risk and emergency funding evaluation only. "
            "You receive a crisis broadcast regarding a data breach. "
            "Your task is to identify the likely financial impact areas, emergency budget needs, and financial stabilization actions. "
            "Think like a crisis finance controller. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, financial_priority, potential_cost_areas, finance_note. "
            "Rules: "
            "immediate_actions should be a short list of finance actions; "
            "financial_priority should describe urgency in plain language; "
            "potential_cost_areas should identify likely impact categories; "
            "finance_note should explain the financial response logic in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following broadcast as Agent 4: Finance Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Your responsibilities:
1. Identify immediate finance actions.
2. Estimate the likely financial exposure categories.
3. Recommend emergency fund or insurance-related actions.
4. Provide a concise finance response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 5: Legal Agent] Assessing compliance, reporting, and liability obligations...")

        system_prompt = (
            "You are Agent 5: Legal Agent in a crisis response multi-agent system. "
            "Your responsibility is legal and regulatory response only. "
            "You receive a crisis broadcast regarding a data breach. "
            "Your task is to identify legal obligations, compliance reporting requirements, notification duties, and liability concerns. "
            "Think like senior corporate counsel during a cyber incident. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, legal_priority, legal_exposure, legal_note. "
            "Rules: "
            "immediate_actions should be a short list of legal response actions; "
            "legal_priority should reflect urgency in plain language; "
            "legal_exposure should summarize the legal and compliance risk level; "
            "legal_note should explain the legal response logic in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following broadcast as Agent 5: Legal Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Your responsibilities:
1. Identify immediate legal and compliance actions.
2. Review possible reporting or notification duties.
3. Assess liability and regulatory exposure.
4. Provide a concise legal response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HRAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 6: HR Agent] Preparing employee support and internal handling guidance...")

        system_prompt = (
            "You are Agent 6: HR Agent in a crisis response multi-agent system. "
            "Your responsibility is people coordination and employee support only. "
            "You receive a crisis broadcast regarding a data breach. "
            "Your task is to determine how employees should be briefed, how customer-facing staff should be guided, "
            "and what morale or organizational support is needed. "
            "Think like an HR crisis response leader. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, hr_priority, employee_focus_areas, hr_note. "
            "Rules: "
            "immediate_actions should be a short list of HR actions; "
            "hr_priority should describe urgency in plain language; "
            "employee_focus_areas should identify internal people-management priorities; "
            "hr_note should explain the HR response logic in 3 to 5 lines. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following broadcast as Agent 6: HR Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Your responsibilities:
1. Define immediate employee communication and support actions.
2. Identify which employee groups need guidance first.
3. Recommend how internal handling should be stabilized.
4. Provide a concise HR response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def integrate(self, broadcast_data: Dict[str, Any], agent_responses: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 7: Decision Agent] Integrating all agent responses into the final crisis response plan...")

        system_prompt = (
            "You are Agent 7: Decision Agent in a crisis response multi-agent system. "
            "Your responsibility is final coordination and integrated decision making. "
            "You receive the original broadcast and all specialist agent responses. "
            "Your task is to produce one unified crisis response plan for executive and operational use. "
            "You must synthesize security, communication, finance, legal, and HR actions into one coherent response. "
            "Think like a chief crisis coordinator. "
            "Return only valid JSON with exactly these keys: "
            "integrated_response_plan, priority_actions, coordination_summary, final_status. "
            "Rules: "
            "integrated_response_plan should summarize the full coordinated plan in 4 to 7 lines; "
            "priority_actions should be a short ordered list of the most critical next steps; "
            "coordination_summary should explain how the organization is responding as a whole in 3 to 5 lines; "
            "final_status should be a concise final state label. "
            "Strictly return only valid JSON. Do not include markdown, explanations outside JSON, or extra keys."
        )

        user_prompt = f"""
You are processing the following incident as Agent 7: Decision Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Specialist Agent Responses:
{json.dumps(agent_responses, indent=2)}

Your responsibilities:
1. Review all specialist responses together.
2. Build one integrated crisis response plan.
3. Identify the highest-priority actions.
4. Summarize the coordinated organizational response.
5. Produce a final crisis response status.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CrisisResponseBroadcastSystem:
    def __init__(self):
        self.coordinator = CrisisCoordinatorAgent()
        self.it_security = ITSecurityAgent()
        self.communications = CommunicationsAgent()
        self.finance = FinanceAgent()
        self.legal = LegalAgent()
        self.hr = HRAgent()
        self.decision = DecisionAgent()

    def run(self, incident_data: Dict[str, Any]) -> Dict[str, Any]:
        broadcast_data = self.coordinator.broadcast(incident_data)

        responses = {
            "it_security_response": self.it_security.respond(broadcast_data),
            "communications_response": self.communications.respond(broadcast_data),
            "finance_response": self.finance.respond(broadcast_data),
            "legal_response": self.legal.respond(broadcast_data),
            "hr_response": self.hr.respond(broadcast_data)
        }

        final_plan = self.decision.integrate(broadcast_data, responses)

        return {
            "broadcast_output": broadcast_data,
            "agent_responses": responses,
            "final_crisis_plan": final_plan
        }


incident_data = {
    "incident_title": "Customer Database Data Breach",
    "incident_message": "Data breach detected in customer database. Immediate response required.",
    "incident_severity": "Critical",
    "affected_assets": [
        "Customer database",
        "Application servers",
        "Customer contact records"
    ],
    "business_context": {
        "company_type": "Digital Services Enterprise",
        "customer_impact_risk": "High",
        "regulatory_exposure": "High",
        "public_reputation_risk": "High"
    }
}

system = CrisisResponseBroadcastSystem()
final_output = system.run(incident_data)

print("\n" + "=" * 74)
print("CRISIS RESPONSE BROADCAST MULTI-AGENT SYSTEM OUTPUT")
print("=" * 74)

print("\n1. Crisis Coordinator Broadcast Output")
print(json.dumps(final_output["broadcast_output"], indent=2))

print("\n2. IT Security Agent Output")
print(json.dumps(final_output["agent_responses"]["it_security_response"], indent=2))

print("\n3. Communications Agent Output")
print(json.dumps(final_output["agent_responses"]["communications_response"], indent=2))

print("\n4. Finance Agent Output")
print(json.dumps(final_output["agent_responses"]["finance_response"], indent=2))

print("\n5. Legal Agent Output")
print(json.dumps(final_output["agent_responses"]["legal_response"], indent=2))

print("\n6. HR Agent Output")
print(json.dumps(final_output["agent_responses"]["hr_response"], indent=2))

print("\n7. Decision Agent Final Crisis Plan")
print(json.dumps(final_output["final_crisis_plan"], indent=2))

print("\n" + "=" * 74)
print("FINAL SYSTEM SUMMARY")
print("=" * 74)
print("This is a broadcast-style multiple-agent crisis response system.")
print("Agent 1 broadcasted the incident to all specialist agents simultaneously.")
print("Agents 2 to 6 generated specialized responses in parallel domains.")
print("Agent 7 integrated all responses into one final coordinated crisis response plan.")
print("This demonstrates broadcaster-coordinator architecture in a refined multi-agent workflow.")

In [20]:
# GROQ API key removed for security
# Scenario: Crisis Response Broadcast Multi-Agent System
# A company detects a serious data breach in its customer database.
# One coordinator agent broadcasts the crisis message to all specialist agents simultaneously.
# Each specialist agent responds with domain-specific actions.
# A final decision agent integrates all responses into one unified crisis response plan.

# ================================
# AGENT 1: Crisis Coordinator (Broadcaster)
# - Broadcasts the crisis message to all agents simultaneously
# ================================

# ================================
# AGENT 2: IT Security Agent
# - Handles containment, patching, server isolation, and forensic analysis
# ================================

# ================================
# AGENT 3: Communications Agent
# - Handles internal memo, stakeholder messaging, and public communication
# ================================

# ================================
# AGENT 4: Finance Agent
# - Assesses business impact, emergency budget, and insurance considerations
# ================================

# ================================
# AGENT 5: Legal Agent
# - Handles compliance obligations, liability, and regulatory reporting
# ================================

# ================================
# AGENT 6: HR Agent
# - Handles employee guidance, morale, and internal support communication
# ================================

# ================================
# AGENT 7: Decision Agent (Coordinator)
# - Collects all responses
# - Produces the final integrated crisis response plan
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.9,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class CrisisCoordinatorAgent:
    def broadcast(self, incident_data: Dict[str, Any]) -> Dict[str, Any]:
        print("[Agent 1: Crisis Coordinator] Broadcasting incident to all specialist agents...")

        system_prompt = (
            "You are Agent 1: Crisis Coordinator in an enterprise broadcast-based crisis response system. "
            "Your job is to convert a raw cyber incident into a crisp, high-priority enterprise broadcast. "
            "You must frame the incident clearly so that all downstream specialist agents can act in parallel with the same situational awareness. "
            "You are responsible for crisis framing, urgency assignment, affected-area identification, and coordination tone. "
            "You are not responsible for technical remediation, legal reporting, public relations execution, finance decisions, or HR implementation. "
            "Return only valid JSON with exactly these keys: "
            "incident_summary, broadcast_message, priority_level, impacted_areas, coordinator_note. "
            "Make the wording sharp, executive-friendly, and operationally useful. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following incident as Agent 1: Crisis Coordinator.

Incident Data:
{json.dumps(incident_data, indent=2)}

Requirements:
1. Summarize the incident in a way leadership and specialist teams can immediately understand.
2. Write a clean enterprise broadcast message.
3. Identify major impacted areas.
4. Assign urgency properly.
5. Add a short coordination note for downstream teams.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class ITSecurityAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: IT Security Agent] Responding to the cyber incident...")

        system_prompt = (
            "You are Agent 2: IT Security Agent in an enterprise crisis response system. "
            "Your job is technical containment, investigation readiness, and cyber risk stabilization. "
            "You must respond like a senior security incident commander. "
            "Your answer should sound specific, credible, and action-oriented rather than generic. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, technical_priority, security_risk_level, security_note. "
            "The note should clearly explain why these actions matter and what security objective they serve. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 2: IT Security Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Provide the most important immediate technical actions.
2. State operational technical priority.
3. Classify cyber risk severity.
4. Add a concise but specific security response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CommunicationsAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Communications Agent] Preparing crisis communication response...")

        system_prompt = (
            "You are Agent 3: Communications Agent in an enterprise crisis response system. "
            "Your job is to stabilize information flow, reduce confusion, protect trust, and prepare both internal and external messaging. "
            "You must respond like a senior crisis communications strategist. "
            "Make the wording polished, realistic, and suitable for executive review. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, communication_priority, audience_groups, communication_note. "
            "The note should explain how the company should communicate in a disciplined and trust-preserving way. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 3: Communications Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define immediate communication actions.
2. Identify the most important audiences.
3. Indicate communication priority.
4. Provide a strong communication strategy note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinanceAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 4: Finance Agent] Assessing financial impact and emergency support...")

        system_prompt = (
            "You are Agent 4: Finance Agent in an enterprise crisis response system. "
            "Your job is to assess financial exposure, emergency liquidity needs, and cost-control readiness during a cyber incident. "
            "You must think like a crisis finance lead briefing senior management. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, financial_priority, potential_cost_areas, finance_note. "
            "The finance_note should sound executive-ready, practical, and specific. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 4: Finance Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Provide immediate finance actions.
2. Identify the most likely cost areas.
3. Assign financial response priority.
4. Add a concise but strong finance note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 5: Legal Agent] Reviewing compliance, liability, and reporting duties...")

        system_prompt = (
            "You are Agent 5: Legal Agent in an enterprise crisis response system. "
            "Your job is to frame the legal and regulatory response to a cyber incident. "
            "You must identify obligations, reporting exposure, liability concerns, and compliance urgency. "
            "Respond like senior corporate counsel. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, legal_priority, legal_exposure, legal_note. "
            "The legal_note should clearly explain why legal action must move in parallel with technical and communication response. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 5: Legal Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Provide immediate legal actions.
2. Assess exposure and reporting needs.
3. Set legal priority.
4. Add a concise legal response note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HRAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 6: HR Agent] Preparing employee support and workforce guidance...")

        system_prompt = (
            "You are Agent 6: HR Agent in an enterprise crisis response system. "
            "Your job is workforce coordination, employee guidance, and internal stability. "
            "You must respond like a senior HR crisis lead. "
            "Focus on briefing employees, supporting customer-facing teams, reducing confusion, and preserving morale under pressure. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, hr_priority, employee_focus_areas, hr_note. "
            "The hr_note should clearly explain how people operations support overall crisis execution. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 6: HR Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define immediate HR and employee-support actions.
2. Identify the most important employee focus areas.
3. Assign HR response priority.
4. Add a concise but meaningful HR note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def integrate(self, broadcast_data: Dict[str, Any], agent_responses: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 7: Decision Agent] Integrating all responses into one coordinated crisis plan...")

        system_prompt = (
            "You are Agent 7: Decision Agent in an enterprise broadcast-based crisis response system. "
            "Your job is not to repeat each agent independently, but to synthesize all specialist outputs into one unified and well-worded enterprise action plan. "
            "You must think like a chief crisis coordinator presenting a clear coordinated response to leadership and operations teams. "
            "Your output must feel integrated, specific, and understandable to both executives and managers. "
            "Return only valid JSON with exactly these keys: "
            "integrated_response_plan, priority_actions, coordination_summary, final_status. "
            "Rules: "
            "integrated_response_plan must read like one joined-up response strategy, not disconnected bullets; "
            "priority_actions should be ordered and highly practical; "
            "coordination_summary should clearly explain how all departments are aligned; "
            "final_status should be concise, professional, and decision-ready. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following incident as Agent 7: Decision Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Specialist Agent Responses:
{json.dumps(agent_responses, indent=2)}

Requirements:
1. Build one integrated response plan in polished wording.
2. Make the output specific, clear, and easy to understand.
3. Show how technical, legal, financial, communication, and HR actions fit together.
4. Provide a realistic order of priority actions.
5. Summarize overall coordination maturity.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CrisisResponseBroadcastSystem:
    def __init__(self):
        self.coordinator = CrisisCoordinatorAgent()
        self.it_security = ITSecurityAgent()
        self.communications = CommunicationsAgent()
        self.finance = FinanceAgent()
        self.legal = LegalAgent()
        self.hr = HRAgent()
        self.decision = DecisionAgent()

    def run(self, incident_data: Dict[str, Any]) -> Dict[str, Any]:
        broadcast_data = self.coordinator.broadcast(incident_data)

        responses = {
            "it_security_response": self.it_security.respond(broadcast_data),
            "communications_response": self.communications.respond(broadcast_data),
            "finance_response": self.finance.respond(broadcast_data),
            "legal_response": self.legal.respond(broadcast_data),
            "hr_response": self.hr.respond(broadcast_data)
        }

        final_plan = self.decision.integrate(broadcast_data, responses)

        return {
            "broadcast_output": broadcast_data,
            "agent_responses": responses,
            "final_crisis_plan": final_plan
        }


incident_data = {
    "incident_title": "Customer Database Data Breach",
    "incident_message": "Data breach detected in customer database. Immediate response required.",
    "incident_severity": "Critical",
    "affected_assets": [
        "Customer database",
        "Application servers",
        "Customer contact records"
    ],
    "business_context": {
        "company_type": "Digital Services Enterprise",
        "customer_impact_risk": "High",
        "regulatory_exposure": "High",
        "public_reputation_risk": "High"
    }
}

system = CrisisResponseBroadcastSystem()
final_output = system.run(incident_data)

print("\n" + "=" * 74)
print("CRISIS RESPONSE BROADCAST MULTI-AGENT SYSTEM OUTPUT")
print("=" * 74)

print("\n1. Crisis Coordinator Broadcast Output")
print(json.dumps(final_output["broadcast_output"], indent=2))

print("\n2. IT Security Agent Output")
print(json.dumps(final_output["agent_responses"]["it_security_response"], indent=2))

print("\n3. Communications Agent Output")
print(json.dumps(final_output["agent_responses"]["communications_response"], indent=2))

print("\n4. Finance Agent Output")
print(json.dumps(final_output["agent_responses"]["finance_response"], indent=2))

print("\n5. Legal Agent Output")
print(json.dumps(final_output["agent_responses"]["legal_response"], indent=2))

print("\n6. HR Agent Output")
print(json.dumps(final_output["agent_responses"]["hr_response"], indent=2))

print("\n7. Decision Agent Final Crisis Plan")
print(json.dumps(final_output["final_crisis_plan"], indent=2))

print("\n" + "=" * 74)
print("FINAL SYSTEM SUMMARY")
print("=" * 74)
print("""
This system demonstrates a broadcast-oriented multi-agent crisis response architecture built for enterprise-grade incident handling.

At the center of the workflow, the Crisis Coordinator Agent transforms the raw breach detection signal into a structured crisis broadcast. This ensures that all downstream specialist agents begin with the same situational understanding, urgency, and operational framing.

Once the broadcast is issued, the specialist agents respond in parallel rather than waiting in sequence. This is a key architectural strength because cyber incidents require multiple business functions to move at the same time.

The IT Security Agent focuses on technical containment, forensic readiness, and infrastructure stabilization.
The Communications Agent controls narrative flow, stakeholder messaging, and trust management.
The Finance Agent evaluates emergency financial exposure and resource allocation.
The Legal Agent handles compliance, reporting obligations, and liability positioning.
The HR Agent ensures that employees receive guidance, clarity, and operational support.

The Decision Agent then acts as the synthesis layer. Instead of merely collecting raw outputs, it integrates all cross-functional responses into one coordinated crisis response plan that leadership and operations teams can execute with clarity.

This architecture demonstrates:
- broadcast-based coordination
- parallel specialist response
- role-specific decision quality
- centralized synthesis
- structured enterprise incident management

In practical terms, the system shows how an organization can respond to a data breach in a way that is technically sound, operationally aligned, legally aware, financially prepared, and people-focused.

Overall, this is a refined multiple-agent orchestration model for real-time crisis response, where specialized intelligence from different business domains is integrated into one unified enterprise action strategy.
""")

In [21]:
# Scenario: Cloud Product Launch Broadcast Multi-Agent System
# A company is preparing to launch a new cloud-based product in a competitive market.
# One launch coordinator agent broadcasts the launch objective to all specialist agents simultaneously.
# Each specialist agent responds with domain-specific launch actions.
# A final decision agent integrates all responses into one unified product launch plan.

# ================================
# AGENT 1: Launch Coordinator (Broadcaster)
# - Broadcasts the launch message to all agents simultaneously
# ================================

# ================================
# AGENT 2: Market Strategy Agent
# - Evaluates customer demand, positioning, and market opportunity
# ================================

# ================================
# AGENT 3: Cloud Engineering Agent
# - Reviews cloud readiness, platform scalability, and technical launch preparedness
# ================================

# ================================
# AGENT 4: Finance & Pricing Agent
# - Assesses pricing model, launch budget, and ROI outlook
# ================================

# ================================
# AGENT 5: Legal & Compliance Agent
# - Reviews compliance, data protection, contracts, and regulatory readiness
# ================================

# ================================
# AGENT 6: Sales & Enablement Agent
# - Prepares go-to-market readiness, sales support, training, and customer onboarding
# ================================

# ================================
# AGENT 7: Decision Agent (Coordinator)
# - Collects all responses
# - Produces the final integrated product launch plan
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class LaunchCoordinatorAgent:
    def broadcast(self, launch_data: Dict[str, Any]) -> Dict[str, Any]:
        print("[Agent 1: Launch Coordinator] Broadcasting product launch objective to all specialist agents...")

        system_prompt = (
            "You are Agent 1: Launch Coordinator in a broadcast-style cloud product launch multi-agent system. "
            "Your role is to convert the raw product launch objective into a clear strategic broadcast for all specialist agents. "
            "You must frame the launch objective, assign urgency, identify critical business areas, and prepare downstream agents for parallel response. "
            "Return only valid JSON with exactly these keys: "
            "launch_summary, broadcast_message, priority_level, focus_areas, coordinator_note. "
            "Make the wording executive-friendly, clear, and operationally useful. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following product launch case as Agent 1: Launch Coordinator.

Launch Data:
{json.dumps(launch_data, indent=2)}

Requirements:
1. Summarize the launch objective clearly.
2. Write a clean launch broadcast message.
3. Identify the major business and technical focus areas.
4. Assign urgency properly.
5. Add a short coordination note for all downstream agents.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class MarketStrategyAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: Market Strategy Agent] Assessing demand, positioning, and launch opportunity...")

        system_prompt = (
            "You are Agent 2: Market Strategy Agent in a cloud product launch multi-agent system. "
            "Your job is market and positioning analysis only. "
            "You must assess launch opportunity, likely customer demand, positioning angle, and market readiness. "
            "Respond like a senior product marketing strategist. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, market_priority, target_segments, market_note. "
            "The market_note should clearly explain why the launch has opportunity and what positioning approach is strongest. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 2: Market Strategy Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define the most important market actions.
2. Identify priority target segments.
3. Set launch market priority.
4. Add a concise but strong market strategy note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CloudEngineeringAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Cloud Engineering Agent] Reviewing technical readiness, scalability, and platform stability...")

        system_prompt = (
            "You are Agent 3: Cloud Engineering Agent in a cloud product launch multi-agent system. "
            "Your job is technical launch readiness only. "
            "You must assess infrastructure readiness, scalability, deployment stability, observability, security posture, and operational resilience. "
            "Respond like a cloud platform engineering lead. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, engineering_priority, technical_readiness_level, engineering_note. "
            "The engineering_note should explain how ready the cloud platform is for launch and what technical gaps must be addressed first. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 3: Cloud Engineering Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define the top technical launch actions.
2. State engineering priority.
3. Assess technical readiness level.
4. Add a concise and realistic engineering readiness note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinancePricingAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 4: Finance & Pricing Agent] Assessing budget, pricing model, and ROI outlook...")

        system_prompt = (
            "You are Agent 4: Finance & Pricing Agent in a cloud product launch multi-agent system. "
            "Your job is financial planning and pricing evaluation only. "
            "You must assess pricing direction, launch budget priorities, revenue outlook, and ROI feasibility. "
            "Respond like a senior finance and pricing strategist. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, financial_priority, revenue_drivers, finance_note. "
            "The finance_note should explain whether the launch looks commercially sound and what pricing logic best supports adoption. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 4: Finance & Pricing Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define immediate financial and pricing actions.
2. Identify the major revenue drivers.
3. State financial priority.
4. Add a concise but practical finance note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalComplianceAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 5: Legal & Compliance Agent] Reviewing contracts, compliance, and regulatory readiness...")

        system_prompt = (
            "You are Agent 5: Legal & Compliance Agent in a cloud product launch multi-agent system. "
            "Your job is legal and compliance readiness only. "
            "You must assess contractual exposure, regulatory requirements, privacy obligations, and cloud compliance concerns. "
            "Respond like a senior legal and compliance advisor. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, legal_priority, compliance_focus_areas, legal_note. "
            "The legal_note should explain what must be cleared before launch to reduce regulatory and contractual risk. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 5: Legal & Compliance Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define immediate legal and compliance actions.
2. Identify the most important compliance focus areas.
3. Assign legal priority.
4. Add a concise legal readiness note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class SalesEnablementAgent:
    def respond(self, broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 6: Sales & Enablement Agent] Preparing go-to-market support, training, and onboarding readiness...")

        system_prompt = (
            "You are Agent 6: Sales & Enablement Agent in a cloud product launch multi-agent system. "
            "Your job is go-to-market execution readiness only. "
            "You must assess sales enablement, customer onboarding readiness, training needs, and launch execution support. "
            "Respond like a revenue enablement and launch operations lead. "
            "Return only valid JSON with exactly these keys: "
            "immediate_actions, enablement_priority, launch_readiness_areas, enablement_note. "
            "The enablement_note should explain how commercial teams can be prepared to support launch success from day one. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following broadcast as Agent 6: Sales & Enablement Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define immediate sales enablement and launch support actions.
2. Identify the most important readiness areas.
3. Assign enablement priority.
4. Add a concise but practical enablement note.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def integrate(self, broadcast_data: Dict[str, Any], agent_responses: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 7: Decision Agent] Integrating all responses into one coordinated launch plan...")

        system_prompt = (
            "You are Agent 7: Decision Agent in a cloud product launch multi-agent system. "
            "Your job is final coordination and integrated launch planning. "
            "You must synthesize market, engineering, finance, legal, and sales readiness into one clear and well-worded launch plan. "
            "Do not simply repeat each agent. Integrate them into one joined-up strategy. "
            "Return only valid JSON with exactly these keys: "
            "integrated_launch_plan, priority_actions, coordination_summary, final_status. "
            "Rules: "
            "integrated_launch_plan must read like one combined launch strategy; "
            "priority_actions should be ordered and practical; "
            "coordination_summary should explain how all teams are aligned; "
            "final_status should be concise, executive-friendly, and launch-oriented. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following case as Agent 7: Decision Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Specialist Agent Responses:
{json.dumps(agent_responses, indent=2)}

Requirements:
1. Build one integrated launch plan in polished wording.
2. Make the output clear, specific, and easy to understand.
3. Show how market, engineering, finance, legal, and enablement actions fit together.
4. Provide a practical order of priority actions.
5. Summarize final launch readiness.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CloudProductLaunchBroadcastSystem:
    def __init__(self):
        self.coordinator = LaunchCoordinatorAgent()
        self.market = MarketStrategyAgent()
        self.engineering = CloudEngineeringAgent()
        self.finance = FinancePricingAgent()
        self.legal = LegalComplianceAgent()
        self.enablement = SalesEnablementAgent()
        self.decision = DecisionAgent()

    def run(self, launch_data: Dict[str, Any]) -> Dict[str, Any]:
        broadcast_data = self.coordinator.broadcast(launch_data)

        responses = {
            "market_strategy_response": self.market.respond(broadcast_data),
            "cloud_engineering_response": self.engineering.respond(broadcast_data),
            "finance_pricing_response": self.finance.respond(broadcast_data),
            "legal_compliance_response": self.legal.respond(broadcast_data),
            "sales_enablement_response": self.enablement.respond(broadcast_data)
        }

        final_plan = self.decision.integrate(broadcast_data, responses)

        return {
            "broadcast_output": broadcast_data,
            "agent_responses": responses,
            "final_launch_plan": final_plan
        }


launch_data = {
    "launch_title": "Launch New Cloud Product",
    "launch_message": "New cloud platform launch planned for enterprise customers. Immediate cross-functional readiness review required.",
    "launch_priority": "High",
    "product_details": {
        "product_name": "CloudX Platform",
        "category": "Cloud Infrastructure & Analytics",
        "target_market": "Enterprise and mid-market customers in Asia",
        "core_value": "Scalable cloud analytics, faster deployment, and lower infrastructure complexity"
    },
    "business_context": {
        "go_to_market_urgency": "High",
        "technical_scaling_risk": "Medium",
        "compliance_complexity": "High",
        "pricing_sensitivity": "Medium",
        "customer_adoption_goal": "Fast adoption in first two quarters after launch"
    }
}

system = CloudProductLaunchBroadcastSystem()
final_output = system.run(launch_data)

print("\n" + "=" * 74)
print("CLOUD PRODUCT LAUNCH BROADCAST MULTI-AGENT SYSTEM OUTPUT")
print("=" * 74)

print("\n1. Launch Coordinator Broadcast Output")
print(json.dumps(final_output["broadcast_output"], indent=2))

print("\n2. Market Strategy Agent Output")
print(json.dumps(final_output["agent_responses"]["market_strategy_response"], indent=2))

print("\n3. Cloud Engineering Agent Output")
print(json.dumps(final_output["agent_responses"]["cloud_engineering_response"], indent=2))

print("\n4. Finance & Pricing Agent Output")
print(json.dumps(final_output["agent_responses"]["finance_pricing_response"], indent=2))

print("\n5. Legal & Compliance Agent Output")
print(json.dumps(final_output["agent_responses"]["legal_compliance_response"], indent=2))

print("\n6. Sales & Enablement Agent Output")
print(json.dumps(final_output["agent_responses"]["sales_enablement_response"], indent=2))

print("\n7. Decision Agent Final Launch Plan")
print(json.dumps(final_output["final_launch_plan"], indent=2))

print("\n" + "=" * 74)
print("FINAL SYSTEM SUMMARY")
print("=" * 74)
print("""
This system demonstrates a broadcast-oriented multi-agent architecture for a cloud product launch.

At the center of the workflow, the Launch Coordinator Agent converts the raw launch objective into a structured broadcast so that every specialist team begins with the same strategic context, urgency level, and execution focus.

Once the broadcast is issued, all specialist agents respond in parallel. This reflects real-world launch environments where go-to-market, engineering, finance, legal, and enablement functions must move together rather than in slow sequential handoffs.

The Market Strategy Agent evaluates demand, customer segments, and positioning strength.
The Cloud Engineering Agent assesses platform readiness, scalability, and technical risk.
The Finance & Pricing Agent evaluates commercial viability, pricing direction, and revenue logic.
The Legal & Compliance Agent reviews launch readiness from a compliance and contractual risk perspective.
The Sales & Enablement Agent prepares training, onboarding, and execution support for commercial teams.

Finally, the Decision Agent acts as the synthesis layer. It integrates all specialist viewpoints into one coordinated launch strategy that leadership can use to decide how, when, and under what conditions the product should go to market.

This design demonstrates broadcast-based coordination, parallel expert response, centralized synthesis, and cross-functional launch planning in a refined multi-agent system.
""")

In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.8 MB/s eta 0:00:00


In [ ]:
# YOUR_GROQ_API_KEY_REMOVED+*

In [4]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market.
# The Manager Agent oversees the process and delegates tasks to specialized workers.

# ================================
# MANAGER AGENT
# - Receives the overall goal
# - Dynamically assigns tasks to worker agents depending on what is needed
# ================================

# ================================
# WORKER AGENTS
# - Market Research Worker
# - Finance Worker
# - Operations Worker
# - Legal Worker
# - HR Worker
# ================================

import os
import json
import re
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1400
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


class ManagerAgent:
    def create_plan(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("[Manager Agent] Reviewing strategic objective and assigning worker tasks...")

        system_prompt = (
            "You are the Manager Agent in a corporate market research and strategy multi-agent system. "
            "Your role is dynamic orchestration and task delegation. "
            "You receive the overall business goal and product context. "
            "You must decide which worker agents should be activated based on the strategic need. "
            "Available workers are: Market Research Worker, Finance Worker, Operations Worker, Legal Worker, HR Worker. "
            "If market demand, competition, segmentation, or customer behavior matters, assign Market Research Worker. "
            "If budget, pricing, investment, ROI, or financial feasibility matters, assign Finance Worker. "
            "If supply chain, logistics, production capacity, or operational execution matters, assign Operations Worker. "
            "If regulations, certifications, intellectual property, contracts, import rules, or compliance matter, assign Legal Worker. "
            "If staffing, hiring, workforce readiness, or training matters, assign HR Worker. "
            "Return only valid JSON with exactly these keys: "
            "goal_summary, assigned_workers, planning_rationale, manager_notes. "
            "Rules: "
            "goal_summary must explain the strategic objective clearly in 2 to 4 lines; "
            "assigned_workers must be a list containing worker names exactly as defined; "
            "planning_rationale must explain why those workers are needed in 4 to 6 lines; "
            "manager_notes must provide concise execution guidance for the downstream workflow. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Manager Agent.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Understand the business objective clearly.
2. Decide which worker agents are required.
3. Build a practical delegation plan.
4. Make sure the plan covers the most important launch feasibility dimensions.
5. Keep the answer executive-friendly and operationally useful.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)

    def final_decision(self, goal: str, product_data: Dict[str, Any], worker_reports: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Manager Agent] Consolidating worker reports into final strategic recommendation...")

        system_prompt = (
            "You are the Manager Agent in a corporate market research and strategy multi-agent system. "
            "Your role now is final synthesis and strategic decision-making. "
            "You receive the overall goal, product context, and all worker reports. "
            "You must integrate market, finance, operations, legal, and HR insights into one final recommendation about product launch feasibility. "
            "Your output should feel like a polished management review summary. "
            "Return only valid JSON with exactly these keys: "
            "launch_feasibility, key_strengths, key_risks, final_recommendation, execution_strategy. "
            "Rules: "
            "launch_feasibility must be one of Not Feasible, Feasible with Conditions, or Highly Feasible; "
            "key_strengths must be a short list of the strongest positive factors; "
            "key_risks must be a short list of the major concerns or launch blockers; "
            "final_recommendation must be 4 to 7 lines and clearly state whether the product should launch and under what conditions; "
            "execution_strategy must explain how the company should move forward in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Manager Agent for final decision-making.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Worker Reports:
{json.dumps(worker_reports, indent=2)}

Requirements:
1. Review all worker reports together.
2. Assess whether launch is feasible.
3. Highlight the strongest opportunities.
4. Highlight the biggest risks.
5. Provide a practical and well-worded final recommendation.
6. Suggest a realistic execution strategy.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class MarketResearchWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Market Research Worker] Collecting competitor data, customer preferences, and demand forecasts...")

        system_prompt = (
            "You are the Market Research Worker in a corporate strategy multi-agent system. "
            "Your role is market intelligence only. "
            "You must evaluate customer demand, market attractiveness, competition intensity, and customer expectations for a new product launch. "
            "Think like a senior market research analyst. "
            "Return only valid JSON with exactly these keys: "
            "demand_forecast, competition_level, customer_preferences, market_insight. "
            "Rules: "
            "demand_forecast must summarize expected demand clearly; "
            "competition_level must describe the competitive environment; "
            "customer_preferences must be a short list of key customer preferences; "
            "market_insight must explain the market opportunity in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Market Research Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate demand direction.
2. Assess competition level.
3. Identify important customer preferences.
4. Provide a practical market insight for launch feasibility.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class FinanceWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Finance Worker] Analyzing budget, ROI, and pricing strategy...")

        system_prompt = (
            "You are the Finance Worker in a corporate strategy multi-agent system. "
            "Your role is financial feasibility only. "
            "You must evaluate likely investment, ROI timeline, pricing approach, and financial risk for the product launch. "
            "Think like a finance strategy lead preparing a launch feasibility memo. "
            "Return only valid JSON with exactly these keys: "
            "estimated_investment, expected_roi_period, pricing_strategy, financial_risk, financial_insight. "
            "Rules: "
            "estimated_investment must provide a realistic high-level investment estimate; "
            "expected_roi_period must estimate likely return period; "
            "pricing_strategy must describe a sensible pricing approach; "
            "financial_risk must be Low, Medium, or High; "
            "financial_insight must explain whether the launch is financially attractive in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Finance Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate the investment scale.
2. Estimate the likely ROI horizon.
3. Suggest an appropriate pricing strategy.
4. Assess the overall financial risk.
5. Provide a concise finance insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class OperationsWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Operations Worker] Evaluating production capacity, supply chain, and logistics...")

        system_prompt = (
            "You are the Operations Worker in a corporate strategy multi-agent system. "
            "Your role is operational feasibility only. "
            "You must evaluate production readiness, supply chain stability, logistics practicality, and operational scale-up capability. "
            "Think like a senior operations strategist. "
            "Return only valid JSON with exactly these keys: "
            "production_capacity, supply_chain_status, logistics_challenge, operations_insight. "
            "Rules: "
            "production_capacity must describe whether the business can scale supply; "
            "supply_chain_status must summarize sourcing and operational stability; "
            "logistics_challenge must identify the main execution difficulty; "
            "operations_insight must explain operational launch readiness in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Operations Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Evaluate operational readiness.
2. Review supply chain status.
3. Identify the main logistics challenge.
4. Provide a practical operations insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class LegalWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Legal Worker] Reviewing compliance, intellectual property, and regional regulations...")

        system_prompt = (
            "You are the Legal Worker in a corporate strategy multi-agent system. "
            "Your role is legal and regulatory feasibility only. "
            "You must assess trademark availability, regulatory obligations, certification needs, import or market-entry restrictions, and compliance risk. "
            "Think like senior legal counsel supporting market entry strategy. "
            "Return only valid JSON with exactly these keys: "
            "trademark_status, compliance_status, legal_risk, legal_insight. "
            "Rules: "
            "trademark_status must indicate whether the intellectual property path appears clear; "
            "compliance_status must summarize regulatory or certification requirements; "
            "legal_risk must be Low, Medium, or High; "
            "legal_insight must explain launch readiness from a legal standpoint in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the Legal Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Review trademark or IP readiness.
2. Identify compliance and regulatory obligations.
3. Assess legal risk.
4. Provide a concise legal insight for launch readiness.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HRWorker:
    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[HR Worker] Assessing staffing needs and training requirements...")

        system_prompt = (
            "You are the HR Worker in a corporate strategy multi-agent system. "
            "Your role is workforce readiness only. "
            "You must evaluate staffing scale, hiring priorities, and training readiness needed to support a product launch. "
            "Think like an HR planning leader supporting expansion. "
            "Return only valid JSON with exactly these keys: "
            "new_hires_required, roles_needed, training_need, hr_insight. "
            "Rules: "
            "new_hires_required must estimate likely hiring scale; "
            "roles_needed must list the most important roles; "
            "training_need must describe key training requirements; "
            "hr_insight must explain whether workforce readiness supports launch in 3 to 5 lines. "
            "Strictly return only valid JSON and no extra text."
        )

        user_prompt = f"""
Process the following business case as the HR Worker.

Strategic Goal:
{goal}

Business Context:
{json.dumps(product_data, indent=2)}

Requirements:
1. Estimate hiring requirements.
2. Identify critical roles needed for launch.
3. Assess training needs.
4. Provide a concise HR readiness insight.

Return only valid JSON.
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class CorporateStrategySystem:
    def __init__(self):
        self.manager = ManagerAgent()
        self.market_worker = MarketResearchWorker()
        self.finance_worker = FinanceWorker()
        self.operations_worker = OperationsWorker()
        self.legal_worker = LegalWorker()
        self.hr_worker = HRWorker()

    def run(self, goal: str, product_data: Dict[str, Any]) -> Dict[str, Any]:
        manager_plan = self.manager.create_plan(goal, product_data)
        assigned_workers = manager_plan.get("assigned_workers", [])

        worker_reports = {}

        for worker_name in assigned_workers:
            if worker_name == "Market Research Worker":
                worker_reports["market_research_report"] = self.market_worker.run(goal, product_data)

            elif worker_name == "Finance Worker":
                worker_reports["finance_report"] = self.finance_worker.run(goal, product_data)

            elif worker_name == "Operations Worker":
                worker_reports["operations_report"] = self.operations_worker.run(goal, product_data)

            elif worker_name == "Legal Worker":
                worker_reports["legal_report"] = self.legal_worker.run(goal, product_data)

            elif worker_name == "HR Worker":
                worker_reports["hr_report"] = self.hr_worker.run(goal, product_data)

        final_decision = self.manager.final_decision(goal, product_data, worker_reports)

        return {
            "manager_plan": manager_plan,
            "worker_reports": worker_reports,
            "manager_final_decision": final_decision
        }


goal = "Evaluate feasibility of launching Product Y in Asia"

product_data = {
    "product_name": "Product Y",
    "target_region": "Asia",
    "industry": "Consumer Technology",
    "budget_unclear": True,
    "regulations_complex": True,
    "production_ready": True,
    "staffing_expansion_needed": True,
    "target_customers": "Urban and digitally active middle-income consumers",
    "launch_objective": "Establish strong presence in high-demand Asian metro markets first"
}

system = CorporateStrategySystem()
final_output = system.run(goal, product_data)

print("\n" + "=" * 72)
print("CORPORATE MARKET RESEARCH & STRATEGY OUTPUT")
print("=" * 72)

print("\n1. Manager Planning Output")
print(json.dumps(final_output["manager_plan"], indent=2))

if "market_research_report" in final_output["worker_reports"]:
    print("\n2. Market Research Worker Output")
    print(json.dumps(final_output["worker_reports"]["market_research_report"], indent=2))

if "finance_report" in final_output["worker_reports"]:
    print("\n3. Finance Worker Output")
    print(json.dumps(final_output["worker_reports"]["finance_report"], indent=2))

if "operations_report" in final_output["worker_reports"]:
    print("\n4. Operations Worker Output")
    print(json.dumps(final_output["worker_reports"]["operations_report"], indent=2))

if "legal_report" in final_output["worker_reports"]:
    print("\n5. Legal Worker Output")
    print(json.dumps(final_output["worker_reports"]["legal_report"], indent=2))

if "hr_report" in final_output["worker_reports"]:
    print("\n6. HR Worker Output")
    print(json.dumps(final_output["worker_reports"]["hr_report"], indent=2))

print("\n7. Manager Final Decision")
print(json.dumps(final_output["manager_final_decision"], indent=2))

print("\n" + "=" * 72)
print("FINAL SYSTEM SUMMARY")
print("=" * 72)
print("""
This system demonstrates a manager-worker multi-agent architecture for corporate market research and launch strategy evaluation.

At the center of the workflow, the Manager Agent receives the strategic objective and dynamically decides which worker agents should be activated. This makes the system adaptive rather than rigid, because delegation changes based on business context such as budget uncertainty, regulatory complexity, operational readiness, and staffing needs.

Once the plan is created, specialist workers operate in their own domains.
The Market Research Worker studies demand, competition, and customer preferences.
The Finance Worker evaluates investment logic, ROI outlook, and pricing direction.
The Operations Worker reviews execution readiness, supply chain capacity, and logistics practicality.
The Legal Worker assesses compliance, certifications, and intellectual property exposure.
The HR Worker evaluates hiring scale, role needs, and training readiness.

After all specialized reports are produced, the Manager Agent synthesizes them into one final strategic recommendation. This means the system does not stop at isolated analysis; it converts multi-domain intelligence into a practical launch decision.

This architecture demonstrates dynamic delegation, role-specialized workers, structured reporting, and centralized strategy synthesis in a refined multi-agent workflow.
""")

In [6]:
# Scenario: Corporate Product Launch Broadcast
# A company is preparing to launch Product X in Q3.
# The Coordinator Agent broadcasts the launch announcement to all departments simultaneously.
# Each department agent responds from its own domain using the Groq API.
# The Decision Agent integrates all responses into one final corporate launch plan.

import os
import json
import re
import asyncio
from typing import Dict, Any
from groq import AsyncGroq

# ================================
# CONFIG
# ================================
# Best practice:
# os.environ["GROQ_API_KEY"] = "your_api_key_here"
# Or set it in your notebook environment before running.

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_NEW_GROQ_API_KEY_HERE")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_NEW_GROQ_API_KEY_HERE":
    raise ValueError("Set a valid GROQ_API_KEY before running this notebook.")

client = AsyncGroq(api_key=GROQ_API_KEY)


# ================================
# SHARED HELPERS
# ================================
async def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = await client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_completion_tokens=1200,
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


# ================================
# AGENT 1: COORDINATOR AGENT
# ================================
async def coordinator_agent(product_name: str, quarter: str, target_market: str, budget: str) -> Dict[str, Any]:
    print("[Coordinator Agent] Broadcasting launch announcement to all departments...")

    launch_data = {
        "product_name": product_name,
        "launch_timeline": quarter,
        "target_market": target_market,
        "budget": budget,
        "message": f"{product_name} launch in {quarter}, target market {target_market}, budget {budget}."
    }

    system_prompt = (
        "You are the Coordinator Agent in a corporate product launch broadcast system. "
        "Your role is to translate the raw launch brief into a clear executive broadcast that all departments can act on simultaneously. "
        "You are not responsible for detailed planning by any single department. "
        "Return only valid JSON with exactly these keys: "
        "launch_summary, broadcast_message, launch_priority, key_focus_areas, coordinator_note. "
        "Rules: "
        "launch_summary must be concise and clear; "
        "broadcast_message must sound like an official cross-functional launch announcement; "
        "launch_priority must be one of Medium, High, or Critical; "
        "key_focus_areas must be a short list of major launch focus themes; "
        "coordinator_note must guide downstream departments. "
        "Do not add any extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this launch case as the Coordinator Agent.

Launch Data:
{json.dumps(launch_data, indent=2)}

Requirements:
1. Summarize the launch clearly.
2. Write a broadcast message for all departments.
3. Identify key focus areas.
4. Assign launch priority.
5. Add a coordinator note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return safe_json_parse(result)


# ================================
# AGENT 2: MARKETING AGENT
# ================================
async def marketing_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Marketing Agent] Building marketing strategy...")

    system_prompt = (
        "You are the Marketing Agent in a corporate product launch system. "
        "Your role is marketing strategy only. "
        "You must define positioning, campaign direction, customer messaging, and marketing channels for the launch. "
        "Return only valid JSON with exactly these keys: "
        "positioning, campaigns, channels, marketing_note. "
        "Rules: "
        "positioning must be 1 to 2 lines; "
        "campaigns must be a short list; "
        "channels must be a short list; "
        "marketing_note must explain the go-to-market approach in simple business language. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Marketing Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define positioning.
2. Suggest campaign ideas.
3. Suggest launch channels.
4. Provide a concise marketing note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Marketing Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 3: FINANCE AGENT
# ================================
async def finance_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Finance Agent] Preparing financial strategy...")

    system_prompt = (
        "You are the Finance Agent in a corporate product launch system. "
        "Your role is budget allocation, commercial risk review, and ROI estimation. "
        "Return only valid JSON with exactly these keys: "
        "budget_allocation, estimated_roi_timeline, forecast, finance_note. "
        "Rules: "
        "budget_allocation must be an object with sensible categories; "
        "estimated_roi_timeline must be short; "
        "forecast must be a one-line financial outlook; "
        "finance_note must explain financial logic in practical business wording. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Finance Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest a launch budget allocation.
2. Estimate ROI timeline.
3. Give a short financial forecast.
4. Add a concise finance note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Finance Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 4: OPERATIONS AGENT
# ================================
async def operations_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Operations Agent] Reviewing production and supply chain readiness...")

    system_prompt = (
        "You are the Operations Agent in a corporate product launch system. "
        "Your role is operational readiness, production scaling, and supply chain execution. "
        "Return only valid JSON with exactly these keys: "
        "production_plan, supply_chain_actions, logistics_status, operations_note. "
        "Rules: "
        "production_plan must be a concise statement; "
        "supply_chain_actions must be a short list; "
        "logistics_status must be a short readiness statement; "
        "operations_note must explain operational readiness clearly. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Operations Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest a production plan.
2. Suggest supply chain actions.
3. State logistics readiness.
4. Add a concise operations note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Operations Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 5: LEGAL AGENT
# ================================
async def legal_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Legal Agent] Checking compliance and contractual readiness...")

    system_prompt = (
        "You are the Legal Agent in a corporate product launch system. "
        "Your role is legal readiness, compliance review, and contract validation. "
        "Return only valid JSON with exactly these keys: "
        "compliance_checks, contract_actions, legal_status, legal_note. "
        "Rules: "
        "compliance_checks must be a short list; "
        "contract_actions must be a short list; "
        "legal_status must be a concise readiness statement; "
        "legal_note must explain what needs to be completed before launch. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Legal Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest compliance checks.
2. Suggest contract actions.
3. State legal readiness status.
4. Add a concise legal note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Legal Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 6: HR AGENT
# ================================
async def hr_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[HR Agent] Preparing staffing and training plans...")

    system_prompt = (
        "You are the HR Agent in a corporate product launch system. "
        "Your role is staffing readiness, hiring support, and training planning. "
        "Return only valid JSON with exactly these keys: "
        "staffing_plan, training_plan, hr_status, hr_note. "
        "Rules: "
        "staffing_plan must be concise; "
        "training_plan must be a short list; "
        "hr_status must be a short readiness statement; "
        "hr_note must explain how workforce preparation supports the launch. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the HR Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest staffing plan.
2. Suggest training plan.
3. State HR readiness.
4. Add a concise HR note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "HR Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 7: DECISION AGENT
# ================================
async def decision_agent(
    broadcast_data: Dict[str, Any],
    marketing_output: Dict[str, Any],
    finance_output: Dict[str, Any],
    operations_output: Dict[str, Any],
    legal_output: Dict[str, Any],
    hr_output: Dict[str, Any]
) -> Dict[str, Any]:
    print("[Decision Agent] Integrating department responses into final corporate launch plan...")

    system_prompt = (
        "You are the Decision Agent in a corporate product launch broadcast system. "
        "Your role is final cross-functional synthesis. "
        "You must integrate all department responses into one polished launch plan. "
        "Do not just repeat department outputs. "
        "Return only valid JSON with exactly these keys: "
        "strategy, priority_actions, launch_readiness_status, integration_note. "
        "Rules: "
        "strategy must read like one joined-up cross-functional launch strategy; "
        "priority_actions must be an ordered short list; "
        "launch_readiness_status must be a concise final readiness status; "
        "integration_note must explain how all departments align into one launch program. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process these department outputs as the Decision Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Department Inputs:
{json.dumps({
    "marketing": marketing_output["response"],
    "finance": finance_output["response"],
    "operations": operations_output["response"],
    "legal": legal_output["response"],
    "hr": hr_output["response"]
}, indent=2)}

Requirements:
1. Build one integrated launch strategy.
2. Provide ordered priority actions.
3. State launch readiness status.
4. Add a concise integration note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    decision = safe_json_parse(result)

    final_plan = {
        "launch_summary": {
            "product_name": broadcast_data.get("product_name", "Unknown"),
            "launch_timeline": broadcast_data.get("launch_timeline", "Unknown"),
            "target_market": broadcast_data.get("target_market", "Unknown"),
            "budget": broadcast_data.get("budget", "Unknown")
        },
        "department_inputs": {
            "marketing": marketing_output["response"],
            "finance": finance_output["response"],
            "operations": operations_output["response"],
            "legal": legal_output["response"],
            "hr": hr_output["response"]
        },
        "final_corporate_launch_plan": decision
    }

    return final_plan


# ================================
# MAIN MULTI-AGENT SYSTEM
# - Uses asyncio.gather() to run all department agents in parallel
# ================================
async def corporate_product_launch_system(product_name: str, quarter: str, target_market: str, budget: str):
    broadcast_data = await coordinator_agent(product_name, quarter, target_market, budget)

    marketing_output, finance_output, operations_output, legal_output, hr_output = await asyncio.gather(
        marketing_agent(broadcast_data),
        finance_agent(broadcast_data),
        operations_agent(broadcast_data),
        legal_agent(broadcast_data),
        hr_agent(broadcast_data)
    )

    final_output = await decision_agent(
        broadcast_data,
        marketing_output,
        finance_output,
        operations_output,
        legal_output,
        hr_output
    )

    return final_output


# ================================
# NOTEBOOK/COLAB RUN
# ================================
async def main():
    result = await corporate_product_launch_system(
        product_name="Product X",
        quarter="Q3",
        target_market="North America",
        budget="$5M"
    )

    print("\n" + "=" * 74)
    print("CORPORATE PRODUCT LAUNCH BROADCAST MULTI-AGENT SYSTEM OUTPUT")
    print("=" * 74)
    print(json.dumps(result, indent=2))

    print("\n" + "=" * 74)
    print("FINAL SYSTEM SUMMARY")
    print("=" * 74)
    print("""
This is a broadcast-style corporate product launch multi-agent system.

The Coordinator Agent creates one launch announcement and shares the same strategic context with all departments at once.
Marketing, Finance, Operations, Legal, and HR then respond in parallel using asyncio.gather(), which makes the workflow faster and closer to a real cross-functional launch environment.

Each department contributes its own specialized view:
- Marketing defines positioning, campaigns, and channels.
- Finance evaluates budget deployment and ROI logic.
- Operations reviews production and supply-chain readiness.
- Legal checks compliance and contractual readiness.
- HR prepares staffing and training support.

Finally, the Decision Agent synthesizes all department outputs into one integrated launch strategy instead of leaving them as isolated recommendations.
This demonstrates parallel coordination, role specialization, and centralized decision synthesis in one multi-agent launch workflow.
""")


# In Jupyter/Colab, run this:
await main()

# In a normal .py file, comment the line above and use:
# if __name__ == "__main__":
#     asyncio.run(main())

[Coordinator Agent] Broadcasting launch announcement to all departments...
[Marketing Agent] Building marketing strategy...
[Finance Agent] Preparing financial strategy...
[Operations Agent] Reviewing production and supply chain readiness...
[Legal Agent] Checking compliance and contractual readiness...
[HR Agent] Preparing staffing and training plans...
[Decision Agent] Integrating department responses into final corporate launch plan...

CORPORATE PRODUCT LAUNCH BROADCAST MULTI-AGENT SYSTEM OUTPUT
{
  "launch_summary": {
    "product_name": "Unknown",
    "launch_timeline": "Unknown",
    "target_market": "Unknown",
    "budget": "Unknown"
  },
  "department_inputs": {
    "marketing": {
      "positioning": "Product X is a cutting-edge solution for the North America market, offering innovative features and benefits that meet the evolving needs of customers.",
      "campaigns": [
        "Influencer Partnerships",
        "Social Media Contests",
        "Targeted Advertising"
     

In [7]:
# Scenario: Corporate Product Launch Broadcast
# A company is preparing to launch Product X in Q3.
# The Coordinator Agent broadcasts the launch announcement to all departments simultaneously.
# Each department agent responds from its own domain using the Groq API.
# The Decision Agent integrates all responses into one final corporate launch plan.

import os
import json
import re
import asyncio
from typing import Dict, Any
from groq import AsyncGroq

# ================================
# CONFIG
# ================================
# Best practice:
# os.environ["GROQ_API_KEY"] = "your_api_key_here"
# Or set it in your notebook environment before running.

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_REMOVED")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_NEW_GROQ_API_KEY_HERE":
    raise ValueError("Set a valid GROQ_API_KEY before running this notebook.")

client = AsyncGroq(api_key=GROQ_API_KEY)


# ================================
# SHARED HELPERS
# ================================
async def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = await client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_completion_tokens=1200,
    )

    content = completion.choices[0].message.content
    if not content or not content.strip():
        raise ValueError("Empty response returned by model")

    return content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON object found. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


# ================================
# AGENT 1: COORDINATOR AGENT
# ================================
async def coordinator_agent(product_name: str, quarter: str, target_market: str, budget: str) -> Dict[str, Any]:
    print("[Coordinator Agent] Broadcasting launch announcement to all departments...")

    launch_data = {
        "product_name": product_name,
        "launch_timeline": quarter,
        "target_market": target_market,
        "budget": budget,
        "message": f"{product_name} launch in {quarter}, target market {target_market}, budget {budget}."
    }

    system_prompt = (
        "You are the Coordinator Agent in a corporate product launch broadcast system. "
        "Your role is to translate the raw launch brief into a clear executive broadcast that all departments can act on simultaneously. "
        "You are not responsible for detailed planning by any single department. "
        "Return only valid JSON with exactly these keys: "
        "launch_summary, broadcast_message, launch_priority, key_focus_areas, coordinator_note. "
        "Rules: "
        "launch_summary must be concise and clear; "
        "broadcast_message must sound like an official cross-functional launch announcement; "
        "launch_priority must be one of Medium, High, or Critical; "
        "key_focus_areas must be a short list of major launch focus themes; "
        "coordinator_note must guide downstream departments. "
        "Do not add any extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this launch case as the Coordinator Agent.

Launch Data:
{json.dumps(launch_data, indent=2)}

Requirements:
1. Summarize the launch clearly.
2. Write a broadcast message for all departments.
3. Identify key focus areas.
4. Assign launch priority.
5. Add a coordinator note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return safe_json_parse(result)


# ================================
# AGENT 2: MARKETING AGENT
# ================================
async def marketing_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Marketing Agent] Building marketing strategy...")

    system_prompt = (
        "You are the Marketing Agent in a corporate product launch system. "
        "Your role is marketing strategy only. "
        "You must define positioning, campaign direction, customer messaging, and marketing channels for the launch. "
        "Return only valid JSON with exactly these keys: "
        "positioning, campaigns, channels, marketing_note. "
        "Rules: "
        "positioning must be 1 to 2 lines; "
        "campaigns must be a short list; "
        "channels must be a short list; "
        "marketing_note must explain the go-to-market approach in simple business language. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Marketing Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Define positioning.
2. Suggest campaign ideas.
3. Suggest launch channels.
4. Provide a concise marketing note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Marketing Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 3: FINANCE AGENT
# ================================
async def finance_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Finance Agent] Preparing financial strategy...")

    system_prompt = (
        "You are the Finance Agent in a corporate product launch system. "
        "Your role is budget allocation, commercial risk review, and ROI estimation. "
        "Return only valid JSON with exactly these keys: "
        "budget_allocation, estimated_roi_timeline, forecast, finance_note. "
        "Rules: "
        "budget_allocation must be an object with sensible categories; "
        "estimated_roi_timeline must be short; "
        "forecast must be a one-line financial outlook; "
        "finance_note must explain financial logic in practical business wording. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Finance Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest a launch budget allocation.
2. Estimate ROI timeline.
3. Give a short financial forecast.
4. Add a concise finance note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Finance Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 4: OPERATIONS AGENT
# ================================
async def operations_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Operations Agent] Reviewing production and supply chain readiness...")

    system_prompt = (
        "You are the Operations Agent in a corporate product launch system. "
        "Your role is operational readiness, production scaling, and supply chain execution. "
        "Return only valid JSON with exactly these keys: "
        "production_plan, supply_chain_actions, logistics_status, operations_note. "
        "Rules: "
        "production_plan must be a concise statement; "
        "supply_chain_actions must be a short list; "
        "logistics_status must be a short readiness statement; "
        "operations_note must explain operational readiness clearly. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Operations Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest a production plan.
2. Suggest supply chain actions.
3. State logistics readiness.
4. Add a concise operations note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Operations Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 5: LEGAL AGENT
# ================================
async def legal_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[Legal Agent] Checking compliance and contractual readiness...")

    system_prompt = (
        "You are the Legal Agent in a corporate product launch system. "
        "Your role is legal readiness, compliance review, and contract validation. "
        "Return only valid JSON with exactly these keys: "
        "compliance_checks, contract_actions, legal_status, legal_note. "
        "Rules: "
        "compliance_checks must be a short list; "
        "contract_actions must be a short list; "
        "legal_status must be a concise readiness statement; "
        "legal_note must explain what needs to be completed before launch. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the Legal Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest compliance checks.
2. Suggest contract actions.
3. State legal readiness status.
4. Add a concise legal note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "Legal Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 6: HR AGENT
# ================================
async def hr_agent(broadcast_data: Dict[str, Any]) -> Dict[str, Any]:
    print("[HR Agent] Preparing staffing and training plans...")

    system_prompt = (
        "You are the HR Agent in a corporate product launch system. "
        "Your role is staffing readiness, hiring support, and training planning. "
        "Return only valid JSON with exactly these keys: "
        "staffing_plan, training_plan, hr_status, hr_note. "
        "Rules: "
        "staffing_plan must be concise; "
        "training_plan must be a short list; "
        "hr_status must be a short readiness statement; "
        "hr_note must explain how workforce preparation supports the launch. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process this broadcast as the HR Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Requirements:
1. Suggest staffing plan.
2. Suggest training plan.
3. State HR readiness.
4. Add a concise HR note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    return {
        "agent": "HR Agent",
        "response": safe_json_parse(result)
    }


# ================================
# AGENT 7: DECISION AGENT
# ================================
async def decision_agent(
    broadcast_data: Dict[str, Any],
    marketing_output: Dict[str, Any],
    finance_output: Dict[str, Any],
    operations_output: Dict[str, Any],
    legal_output: Dict[str, Any],
    hr_output: Dict[str, Any]
) -> Dict[str, Any]:
    print("[Decision Agent] Integrating department responses into final corporate launch plan...")

    system_prompt = (
        "You are the Decision Agent in a corporate product launch broadcast system. "
        "Your role is final cross-functional synthesis. "
        "You must integrate all department responses into one polished launch plan. "
        "Do not just repeat department outputs. "
        "Return only valid JSON with exactly these keys: "
        "strategy, priority_actions, launch_readiness_status, integration_note. "
        "Rules: "
        "strategy must read like one joined-up cross-functional launch strategy; "
        "priority_actions must be an ordered short list; "
        "launch_readiness_status must be a concise final readiness status; "
        "integration_note must explain how all departments align into one launch program. "
        "Do not add extra keys or text outside JSON."
    )

    user_prompt = f"""
Process these department outputs as the Decision Agent.

Broadcast Data:
{json.dumps(broadcast_data, indent=2)}

Department Inputs:
{json.dumps({
    "marketing": marketing_output["response"],
    "finance": finance_output["response"],
    "operations": operations_output["response"],
    "legal": legal_output["response"],
    "hr": hr_output["response"]
}, indent=2)}

Requirements:
1. Build one integrated launch strategy.
2. Provide ordered priority actions.
3. State launch readiness status.
4. Add a concise integration note.

Return only valid JSON.
"""

    result = await call_llm(system_prompt, user_prompt)
    decision = safe_json_parse(result)

    final_plan = {
        "launch_summary": {
            "product_name": broadcast_data.get("product_name", "Unknown"),
            "launch_timeline": broadcast_data.get("launch_timeline", "Unknown"),
            "target_market": broadcast_data.get("target_market", "Unknown"),
            "budget": broadcast_data.get("budget", "Unknown")
        },
        "department_inputs": {
            "marketing": marketing_output["response"],
            "finance": finance_output["response"],
            "operations": operations_output["response"],
            "legal": legal_output["response"],
            "hr": hr_output["response"]
        },
        "final_corporate_launch_plan": decision
    }

    return final_plan


# ================================
# MAIN MULTI-AGENT SYSTEM
# - Uses asyncio.gather() to run all department agents in parallel
# ================================
async def corporate_product_launch_system(product_name: str, quarter: str, target_market: str, budget: str):
    broadcast_data = await coordinator_agent(product_name, quarter, target_market, budget)

    marketing_output, finance_output, operations_output, legal_output, hr_output = await asyncio.gather(
        marketing_agent(broadcast_data),
        finance_agent(broadcast_data),
        operations_agent(broadcast_data),
        legal_agent(broadcast_data),
        hr_agent(broadcast_data)
    )

    final_output = await decision_agent(
        broadcast_data,
        marketing_output,
        finance_output,
        operations_output,
        legal_output,
        hr_output
    )

    return final_output


# ================================
# NOTEBOOK/COLAB RUN
# ================================
async def main():
    result = await corporate_product_launch_system(
        product_name="Product X",
        quarter="Q3",
        target_market="North America",
        budget="$5M"
    )

    print("\n" + "=" * 74)
    print("CORPORATE PRODUCT LAUNCH BROADCAST MULTI-AGENT SYSTEM OUTPUT")
    print("=" * 74)
    print(json.dumps(result, indent=2))

    print("\n" + "=" * 74)
    print("FINAL SYSTEM SUMMARY")
    print("=" * 74)
    print("""
This is a broadcast-style corporate product launch multi-agent system.

The Coordinator Agent creates one launch announcement and shares the same strategic context with all departments at once.
Marketing, Finance, Operations, Legal, and HR then respond in parallel using asyncio.gather(), which makes the workflow faster and closer to a real cross-functional launch environment.

Each department contributes its own specialized view:
- Marketing defines positioning, campaigns, and channels.
- Finance evaluates budget deployment and ROI logic.
- Operations reviews production and supply-chain readiness.
- Legal checks compliance and contractual readiness.
- HR prepares staffing and training support.

Finally, the Decision Agent synthesizes all department outputs into one integrated launch strategy instead of leaving them as isolated recommendations.
This demonstrates parallel coordination, role specialization, and centralized decision synthesis in one multi-agent launch workflow.
""")


# In Jupyter/Colab, run this:
await main()

# In a normal .py file, comment the line above and use:
# if __name__ == "__main__":
#     asyncio.run(main())

In [8]:
# Project Description
# Project Title

# Scalable Multi-Agent AI Research Pipeline for EV Market Intelligence

# Objective

# This project simulates a real-world AI research team that generates a market report on:

# “Write a comprehensive market report on EV industry trends in India for 2025.”

# Agent Roles

# Orchestrator Agent → creates the plan and controls agent flow

# Search Agent → collects EV industry information using the LLM API

# Analyst Agent → interprets findings and extracts strategic insights

# Writer Agent → drafts the full report

# QA Agent → reviews, improves, and finalizes the report

# Shared Memory / Message Bus → enables all agents to read and write intermediate outputs

# Where API Connectivity Happens

# API connectivity is implemented inside the LLMService class:

# self.client = Groq(api_key=api_key) → this creates the Groq API connection

# self.client.chat.completions.create(...) → this is the actual API call

# all agents use the same shared LLM service instead of creating separate API clients

# This makes the project:

# cleaner

# more maintainable

# more scalable

# Why This Version Is Scalable

# centralized API layer

# reusable agent base structure

# shared memory for coordination

# modular pipeline

# easy to add new agents later

# clear separation of orchestration, intelligence, memory, and reporting



import os
import json
import re
from datetime import datetime
from typing import Dict, Any, List, Optional
from groq import Groq


# ============================================================
# CONFIGURATION
# ============================================================
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_NEW_GROQ_API_KEY_HERE":
    raise ValueError("Please provide a valid GROQ_API_KEY before running the project.")


# ============================================================
# API LAYER
# API CONNECTIVITY HAPPENS HERE
# ============================================================
class LLMService:
    def __init__(self, api_key: str, model: str):
        self.model = model

        # API connectivity starts here
        self.client = Groq(api_key=api_key)

    def generate(self, system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
        # Actual API call happens here
        completion = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=1400
        )

        content = completion.choices[0].message.content
        if not content or not content.strip():
            raise ValueError("Empty response received from LLM API")

        return content.strip()


# ============================================================
# JSON PARSER
# ============================================================
def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON found in model output. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


# ============================================================
# SHARED MEMORY / MESSAGE BUS
# ALL AGENTS CONNECT THROUGH THIS
# ============================================================
class SharedMemory:
    def __init__(self):
        self.store: Dict[str, Any] = {}
        self.logs: List[Dict[str, Any]] = []

    def update(self, agent_name: str, result: Any) -> None:
        self.store[agent_name] = result
        self.logs.append({
            "agent": agent_name,
            "timestamp": str(datetime.now()),
            "status": "updated"
        })

    def get(self, agent_name: str) -> Optional[Any]:
        return self.store.get(agent_name)

    def get_all(self) -> Dict[str, Any]:
        return self.store

    def get_logs(self) -> List[Dict[str, Any]]:
        return self.logs


# ============================================================
# BASE AGENT
# ============================================================
class BaseAgent:
    def __init__(self, name: str, llm: LLMService):
        self.name = name
        self.llm = llm

    def save_to_memory(self, memory: SharedMemory, result: Any) -> Any:
        memory.update(self.name, result)
        return result


# ============================================================
# ORCHESTRATOR AGENT
# ============================================================
class OrchestratorAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Orchestrator Agent] Planning and assigning the research workflow...")

        system_prompt = (
            "You are the Orchestrator Agent in a scalable multi-agent AI research pipeline. "
            "Your role is to interpret the goal, split the work into practical stages, "
            "define the execution order, and produce a structured workflow plan. "
            "You do not perform research or writing directly. "
            "Return only valid JSON with exactly these keys: "
            "goal_summary, task_breakdown, execution_order, orchestration_note. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Your responsibilities:
1. Summarize the goal.
2. Break it into logical tasks.
3. Define the execution order of agents.
4. Add an orchestration note for the pipeline.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# SEARCH AGENT
# ============================================================
class SearchAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Search Agent] Collecting EV market signals and research inputs...")

        orchestrator_plan = memory.get("orchestrator")

        system_prompt = (
            "You are the Search Agent in a scalable multi-agent AI research pipeline. "
            "Your role is to collect structured research inputs for an EV market intelligence report. "
            "You must produce realistic market findings, consumer patterns, business drivers, "
            "industry movement, and operational signals in structured form. "
            "Return only valid JSON with exactly these keys: "
            "industry_overview, key_trends, consumer_preferences, market_drivers, market_challenges, competitive_landscape, forecast_signals. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Orchestrator Plan:
{json.dumps(orchestrator_plan, indent=2)}

Your responsibilities:
1. Gather EV industry research findings for India in 2025.
2. Cover market direction, trends, customer preferences, drivers, risks, competition, and forecast signals.
3. Keep the output structured for downstream analysis.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# ANALYST AGENT
# ============================================================
class AnalystAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Analyst Agent] Interpreting findings and extracting strategic insights...")

        search_data = memory.get("search")

        system_prompt = (
            "You are the Analyst Agent in a scalable multi-agent AI research pipeline. "
            "Your role is to transform research findings into strategic business intelligence. "
            "You must identify executive insights, opportunities, risks, strategic meaning, and recommended focus areas. "
            "Return only valid JSON with exactly these keys: "
            "executive_insights, opportunities, risks, strategic_interpretation, recommended_focus_areas. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Your responsibilities:
1. Interpret the research findings.
2. Convert them into executive-level strategic insights.
3. Identify opportunities and risks.
4. Explain the broader business meaning.
5. Recommend strategic focus areas.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# WRITER AGENT
# ============================================================
class WriterAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Writer Agent] Drafting the market report...")

        search_data = memory.get("search")
        analysis_data = memory.get("analyst")

        system_prompt = (
            "You are the Writer Agent in a scalable multi-agent AI research pipeline. "
            "Your role is to draft a clear, professional, structured market report. "
            "You must write in a polished business style suitable for management review. "
            "Return only valid JSON with exactly these keys: "
            "report_title, executive_summary, full_report, writer_note. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Analysis Insights:
{json.dumps(analysis_data, indent=2)}

Your responsibilities:
1. Write a strong report title.
2. Create an executive summary.
3. Draft the full market report in structured business language.
4. Add a short writer note.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.4)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# QA AGENT
# ============================================================
class QAAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[QA Agent] Reviewing report quality, completeness, and business tone...")

        writer_output = memory.get("writer")
        analysis_output = memory.get("analyst")

        system_prompt = (
            "You are the QA Agent in a scalable multi-agent AI research pipeline. "
            "Your role is to review and improve the generated report for clarity, structure, factual plausibility, tone, and completeness. "
            "You must finalize the report in a more polished and decision-friendly form. "
            "Return only valid JSON with exactly these keys: "
            "final_report, qa_improvements, quality_status, qa_note. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Draft Report:
{json.dumps(writer_output, indent=2)}

Analyst Output:
{json.dumps(analysis_output, indent=2)}

Your responsibilities:
1. Review the report for business quality.
2. Improve clarity, structure, and usefulness.
3. Finalize the report.
4. Mention what improvements were made.
5. Set quality status.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.25)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# MAIN SCALABLE PIPELINE
# ============================================================
class AIResearchPipeline:
    def __init__(self, llm_service: LLMService):
        self.memory = SharedMemory()

        # All agents are connected to the same API layer and same shared memory model
        self.orchestrator = OrchestratorAgent("orchestrator", llm_service)
        self.search = SearchAgent("search", llm_service)
        self.analyst = AnalystAgent("analyst", llm_service)
        self.writer = WriterAgent("writer", llm_service)
        self.qa = QAAgent("qa", llm_service)

        self.execution_agents = [
            self.orchestrator,
            self.search,
            self.analyst,
            self.writer,
            self.qa
        ]

    def run(self, goal: str) -> Dict[str, Any]:
        print("=" * 80)
        print("SCALABLE MULTI-AGENT AI RESEARCH PIPELINE")
        print("=" * 80)
        print("Goal:", goal)
        print("=" * 80)

        self.orchestrator.run(goal, self.memory)
        self.search.run(goal, self.memory)
        self.analyst.run(goal, self.memory)
        self.writer.run(goal, self.memory)
        self.qa.run(goal, self.memory)

        return {
            "memory": self.memory.get_all(),
            "logs": self.memory.get_logs()
        }


# ============================================================
# RUN PROJECT
# ============================================================
if __name__ == "__main__":
    llm_service = LLMService(
        api_key=GROQ_API_KEY,
        model=GROQ_MODEL
    )

    goal = "Write a comprehensive market report on EV industry trends in India for 2025."

    pipeline = AIResearchPipeline(llm_service)
    result = pipeline.run(goal)

    final_qa_output = result["memory"]["qa"]

    print("\n" + "=" * 80)
    print("FINAL REPORT")
    print("=" * 80)
    print(final_qa_output["final_report"])

    print("\n" + "=" * 80)
    print("QA IMPROVEMENTS")
    print("=" * 80)
    print(json.dumps(final_qa_output["qa_improvements"], indent=2))

    print("\n" + "=" * 80)
    print("PIPELINE LOGS")
    print("=" * 80)
    print(json.dumps(result["logs"], indent=2))

    print("\n" + "=" * 80)
    print("FINAL SYSTEM SUMMARY")
    print("=" * 80)
    print("""
This project implements a scalable multi-agent AI research architecture for business intelligence generation.

At the foundation of the system, API connectivity is centralized through the LLMService layer. This component creates the Groq API client once and exposes a reusable generate() interface to all agents. Because the API is not hardwired into each agent separately, the design becomes cleaner, easier to maintain, and more scalable.

The connectivity between agents is achieved through SharedMemory, which functions as a message bus. Every agent reads the previous output it needs and writes its own result back into memory. This creates a connected pipeline where intelligence flows stage by stage without tightly coupling the agents to each other.

The Orchestrator Agent defines the workflow.
The Search Agent gathers structured domain findings.
The Analyst Agent interprets those findings into strategic insights.
The Writer Agent converts insights into a professional report.
The QA Agent reviews and strengthens the final deliverable.

This architecture is scalable because:
- the API layer is centralized
- agents are modular and reusable
- shared memory allows loose coupling
- new agents can be added without redesigning the full system
- logging makes the workflow traceable
- the pipeline can later be parallelized or extended into async execution

In practical terms, this project demonstrates how a real AI research team can be simulated through multiple specialized agents working together over shared infrastructure and a common intelligence layer.
""".strip())

In [9]:
import os
import json
import re
from datetime import datetime
from typing import Dict, Any, List, Optional
from groq import Groq


# ============================================================
# CONFIGURATION
# ============================================================
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_NEW_GROQ_API_KEY_HERE":
    raise ValueError("Please provide a valid GROQ_API_KEY before running the project.")


# ============================================================
# CENTRALIZED API LAYER
# API CONNECTIVITY HAPPENS HERE
# ============================================================
class LLMService:
    def __init__(self, api_key: str, model: str):
        self.model = model
        self.client = Groq(api_key=api_key)

    def generate(self, system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
        completion = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=1800
        )

        content = completion.choices[0].message.content
        if not content or not content.strip():
            raise ValueError("Empty response received from LLM API")

        return content.strip()


# ============================================================
# SAFE JSON PARSER
# ============================================================
def safe_json_parse(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No valid JSON found in model output. Raw response: {text}")

        cleaned = match.group(0)
        cleaned = cleaned.replace("\r", " ")
        cleaned = cleaned.replace("\t", " ")
        cleaned = cleaned.replace("\n", " ")

        try:
            return json.loads(cleaned)
        except Exception:
            cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            return json.loads(cleaned)


# ============================================================
# SHARED MEMORY / MESSAGE BUS
# ============================================================
class SharedMemory:
    def __init__(self):
        self.store: Dict[str, Any] = {}
        self.logs: List[Dict[str, Any]] = []

    def update(self, agent_name: str, result: Any) -> None:
        self.store[agent_name] = result
        self.logs.append({
            "agent": agent_name,
            "timestamp": str(datetime.now()),
            "status": "updated"
        })

    def get(self, agent_name: str) -> Optional[Any]:
        return self.store.get(agent_name)

    def get_all(self) -> Dict[str, Any]:
        return self.store

    def get_logs(self) -> List[Dict[str, Any]]:
        return self.logs


# ============================================================
# BASE AGENT
# ============================================================
class BaseAgent:
    def __init__(self, name: str, llm: LLMService):
        self.name = name
        self.llm = llm

    def save_to_memory(self, memory: SharedMemory, result: Any) -> Any:
        memory.update(self.name, result)
        return result


# ============================================================
# ORCHESTRATOR AGENT
# ============================================================
class OrchestratorAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Orchestrator Agent] Planning the EV research workflow...")

        system_prompt = (
            "You are the Orchestrator Agent in a world-class multi-agent EV market research pipeline. "
            "Your role is to interpret the EV research objective, split the work into meaningful stages, "
            "define execution order, and guide downstream agents. "
            "The topic is strictly EV industry trends in India for 2025. "
            "Return only valid JSON with exactly these keys: "
            "goal_summary, task_breakdown, execution_order, orchestration_note, expected_deliverable. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Responsibilities:
1. Summarize the EV research objective.
2. Break the task into practical research stages.
3. Define execution order of agents.
4. Add orchestration guidance.
5. Describe the expected final deliverable.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.2)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# SEARCH AGENT
# ============================================================
class SearchAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Search Agent] Gathering EV market data, signals, and trend inputs...")

        orchestrator_plan = memory.get("orchestrator")

        system_prompt = (
            "You are the Search Agent in a world-class multi-agent EV market research pipeline. "
            "Your role is to collect structured EV industry research inputs for India in 2025. "
            "You must produce realistic, plausible, research-style findings across demand, policy, competition, infrastructure, "
            "consumer behavior, market drivers, constraints, and forecast signals. "
            "Do not write the final report. "
            "Return only valid JSON with exactly these keys: "
            "industry_overview, key_trends, consumer_preferences, market_drivers, market_challenges, "
            "competitive_landscape, charging_infrastructure_outlook, policy_and_regulation_signals, forecast_signals, source_style_note. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Orchestrator Plan:
{json.dumps(orchestrator_plan, indent=2)}

Responsibilities:
1. Collect EV market findings related to India in 2025.
2. Cover demand trends, consumer behavior, policy signals, competition, infrastructure, and constraints.
3. Structure the output so downstream analysis becomes stronger.
4. Keep the findings business-usable and realistic.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.25)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# ANALYST AGENT
# ============================================================
class AnalystAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Analyst Agent] Interpreting EV research data into strategic insights...")

        search_data = memory.get("search")

        system_prompt = (
            "You are the Analyst Agent in a world-class multi-agent EV market research pipeline. "
            "Your role is to transform structured EV research into strategic business intelligence. "
            "You must identify executive insights, segment-level opportunities, critical risks, strategic interpretation, "
            "competitive implications, and recommended focus areas. "
            "Return only valid JSON with exactly these keys: "
            "executive_insights, opportunity_areas, risk_factors, strategic_interpretation, competitive_implications, "
            "recommended_focus_areas, confidence_assessment, analyst_note. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Responsibilities:
1. Interpret the EV research findings.
2. Convert them into executive-level strategy insights.
3. Identify opportunity areas and risk factors.
4. Explain the strategic and competitive meaning.
5. Recommend the most important business focus areas.
6. Provide a confidence assessment for the analysis.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.3)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# WRITER AGENT
# ============================================================
class WriterAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Writer Agent] Drafting the EV market report in a structured professional format...")

        search_data = memory.get("search")
        analysis_data = memory.get("analyst")

        system_prompt = (
            "You are the Writer Agent in a world-class multi-agent EV market research pipeline. "
            "Your role is to draft a polished, structured, management-grade market report on EV industry trends in India for 2025. "
            "You must write clearly, professionally, and in a boardroom-friendly style. "
            "Return only valid JSON with exactly these keys: "
            "report_title, executive_summary, report_sections, draft_report, writer_note. "
            "Rules: "
            "report_sections must be a list of section names; "
            "draft_report must be long-form, descriptive, and complete. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Analyst Output:
{json.dumps(analysis_data, indent=2)}

Responsibilities:
1. Create a strong report title.
2. Write a crisp executive summary.
3. Define report sections.
4. Draft a comprehensive EV market report for India 2025.
5. Add a short writer note.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.45)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# DYNAMIC QA AGENT
# WORLD-CLASS QUALITY CONTROL LAYER
# ============================================================
class QAAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[QA Agent] Running dynamic review: quality, coverage, strategy value, tone, and revision upgrade...")

        writer_output = memory.get("writer")
        analyst_output = memory.get("analyst")
        search_output = memory.get("search")
        orchestrator_output = memory.get("orchestrator")

        system_prompt = (
            "You are the QA Agent in a world-class multi-agent EV market research pipeline. "
            "You are not a basic proofreader. You are a dynamic quality intelligence layer. "
            "Your job is to deeply evaluate the drafted report across multiple dimensions: "
            "coverage completeness, EV domain relevance, strategic usefulness, executive clarity, logical consistency, "
            "factual plausibility, report structure quality, writing tone, and decision-readiness. "
            "You must identify what is strong, what is weak, what is missing, and then produce an improved final report. "
            "Think like a top-tier editorial strategist, research reviewer, and business quality auditor combined. "
            "Return only valid JSON with exactly these keys: "
            "quality_score, quality_dimensions, missing_elements, improvement_actions, revised_executive_summary, "
            "final_report, quality_status, qa_note. "
            "Rules: "
            "quality_score must be a number from 1 to 10; "
            "quality_dimensions must be an object with scores for clarity, completeness, strategic_value, EV_relevance, and executive_readability; "
            "missing_elements must be a list of missing or weak areas; "
            "improvement_actions must be a list of what was improved; "
            "revised_executive_summary must be stronger than the draft version; "
            "final_report must be a revised and improved report, not just the original copied again; "
            "quality_status must be one of Needs Improvement, Good, Very Good, or Boardroom Ready; "
            "qa_note must explain the final quality judgment in strong professional language. "
            "Strictly return only valid JSON."
        )

        user_prompt = f"""
Research Goal:
{goal}

Orchestrator Output:
{json.dumps(orchestrator_output, indent=2)}

Search Output:
{json.dumps(search_output, indent=2)}

Analyst Output:
{json.dumps(analyst_output, indent=2)}

Writer Draft:
{json.dumps(writer_output, indent=2)}

Your dynamic QA responsibilities:
1. Evaluate whether the draft fully addresses EV industry trends in India for 2025.
2. Check if the report is strategically valuable, not just descriptive.
3. Detect missing sections, weak logic, shallow analysis, or generic wording.
4. Evaluate business usefulness for a decision-maker.
5. Improve the executive summary.
6. Produce a final revised report that is stronger than the draft.
7. Give a final quality score and status.

Return only valid JSON.
"""

        result = self.llm.generate(system_prompt, user_prompt, temperature=0.25)
        parsed = safe_json_parse(result)
        return self.save_to_memory(memory, parsed)


# ============================================================
# MAIN WORLD-CLASS PIPELINE
# ============================================================
class WorldClassEVResearchPipeline:
    def __init__(self, llm_service: LLMService):
        self.memory = SharedMemory()

        self.orchestrator = OrchestratorAgent("orchestrator", llm_service)
        self.search = SearchAgent("search", llm_service)
        self.analyst = AnalystAgent("analyst", llm_service)
        self.writer = WriterAgent("writer", llm_service)
        self.qa = QAAgent("qa", llm_service)

        self.pipeline_agents = [
            self.orchestrator,
            self.search,
            self.analyst,
            self.writer,
            self.qa
        ]

    def run(self, goal: str) -> Dict[str, Any]:
        print("=" * 90)
        print("WORLD-CLASS MULTI-AGENT AI RESEARCH TEAM")
        print("=" * 90)
        print("Goal:", goal)
        print("=" * 90)

        self.orchestrator.run(goal, self.memory)
        self.search.run(goal, self.memory)
        self.analyst.run(goal, self.memory)
        self.writer.run(goal, self.memory)
        self.qa.run(goal, self.memory)

        return {
            "memory": self.memory.get_all(),
            "logs": self.memory.get_logs()
        }


# ============================================================
# RUN PROJECT
# ============================================================
if __name__ == "__main__":
    llm_service = LLMService(
        api_key=GROQ_API_KEY,
        model=GROQ_MODEL
    )

    goal = "Write a comprehensive market report on EV industry trends in India for 2025."

    pipeline = WorldClassEVResearchPipeline(llm_service)
    result = pipeline.run(goal)

    final_qa_output = result["memory"]["qa"]

    print("\n" + "=" * 90)
    print("FINAL REVISED REPORT")
    print("=" * 90)
    print(final_qa_output["final_report"])

    print("\n" + "=" * 90)
    print("QUALITY SCORE")
    print("=" * 90)
    print(final_qa_output["quality_score"])

    print("\n" + "=" * 90)
    print("QUALITY DIMENSIONS")
    print("=" * 90)
    print(json.dumps(final_qa_output["quality_dimensions"], indent=2))

    print("\n" + "=" * 90)
    print("MISSING ELEMENTS DETECTED")
    print("=" * 90)
    print(json.dumps(final_qa_output["missing_elements"], indent=2))

    print("\n" + "=" * 90)
    print("IMPROVEMENT ACTIONS TAKEN")
    print("=" * 90)
    print(json.dumps(final_qa_output["improvement_actions"], indent=2))

    print("\n" + "=" * 90)
    print("PIPELINE LOGS")
    print("=" * 90)
    print(json.dumps(result["logs"], indent=2))

    print("\n" + "=" * 90)
    print("FINAL SYSTEM SUMMARY")
    print("=" * 90)
    print("""
This project implements a world-class multi-agent AI research architecture for EV market intelligence generation.

At the foundation of the system, API connectivity is centralized through the LLMService layer. This design avoids scattering API calls across the codebase and creates a reusable intelligence layer that every agent can access. The system is therefore cleaner, easier to extend, and more scalable.

All agents are connected through SharedMemory, which acts as a real-time message bus. Each agent writes structured findings into memory and downstream agents read those outputs to continue the pipeline. This creates a connected research workflow instead of isolated AI tasks.

The Orchestrator Agent converts the high-level business goal into a structured execution plan.
The Search Agent collects realistic EV market signals covering trends, drivers, challenges, competition, infrastructure, and policy direction.
The Analyst Agent transforms research inputs into strategic insights, opportunities, risks, and executive interpretations.
The Writer Agent converts the intelligence into a professional market report suitable for management review.
The QA Agent acts as a dynamic quality engine. Instead of doing surface-level proofreading, it evaluates depth, EV relevance, strategic value, structure, clarity, completeness, and executive usefulness. It detects missing elements, scores quality dimensions, and produces a revised final report that is stronger than the original draft.

This architecture is strong because it combines:
- modular agent specialization
- centralized API intelligence
- shared memory coordination
- structured reporting
- dynamic quality control
- scalable design for future agent expansion

In practical terms, this project simulates how a high-performing AI research team could collaborate to generate a meaningful market intelligence report on EV industry trends in India for 2025.
""".strip())

In [10]:
import json
import re
import time
from datetime import datetime
from typing import Dict, Any, List, Optional
from groq import Groq


# ============================================================
# CONFIGURATION
# ============================================================
GROQ_API_KEY = "32Wy"
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY or GROQ_API_KEY == "YOUR_NEW_GROQ_API_KEY_HERE":
    raise ValueError("Please provide a valid GROQ_API_KEY before running the project.")


# ============================================================
# SAFE JSON PARSER
# ============================================================
def safe_json_parse(text: str) -> Dict[str, Any]:
    if not text or not text.strip():
        raise ValueError("Empty model response")

    text = text.strip()

    # First direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # Remove markdown code fences if present
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Extract the biggest JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No valid JSON object found. Raw response: {text}")

    cleaned = match.group(0)

    # Clean common control chars
    cleaned = cleaned.replace("\r", " ")
    cleaned = cleaned.replace("\t", " ")
    cleaned = re.sub(r"[\x00-\x1F]+", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Remove trailing commas before } or ]
    cleaned = re.sub(r",\s*}", "}", cleaned)
    cleaned = re.sub(r",\s*]", "]", cleaned)

    try:
        return json.loads(cleaned)
    except Exception as e:
        print("\nRAW MODEL OUTPUT:\n", text)
        print("\nCLEANED JSON CANDIDATE:\n", cleaned)
        raise ValueError(f"Model returned invalid JSON even after cleaning: {e}")


# ============================================================
# API LAYER WITH RETRY
# ============================================================
class LLMService:
    def __init__(self, api_key: str, model: str):
        self.model = model
        self.client = Groq(api_key=api_key)

    def generate(self, system_prompt: str, user_prompt: str, temperature: float = 0.3, retries: int = 3) -> str:
        final_prompt = (
            system_prompt
            + " Return only compact valid JSON. "
              "Use double quotes for all keys and string values. "
              "Do not include markdown, comments, explanations, or code fences. "
              "Do not leave trailing commas. "
              "Do not include any text before or after the JSON object."
        )

        last_error = None

        for attempt in range(1, retries + 1):
            try:
                completion = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": final_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_completion_tokens=1400
                )

                content = completion.choices[0].message.content
                if not content or not content.strip():
                    raise ValueError("Empty response received from LLM API")

                # Validate once here itself
                safe_json_parse(content)
                return content.strip()

            except Exception as e:
                last_error = e
                print(f"[LLMService] Attempt {attempt} failed: {e}")
                time.sleep(1)

        raise ValueError(f"LLM failed after {retries} attempts. Last error: {last_error}")


# ============================================================
# SHARED MEMORY
# ============================================================
class SharedMemory:
    def __init__(self):
        self.store: Dict[str, Any] = {}
        self.logs: List[Dict[str, Any]] = []

    def update(self, agent_name: str, result: Any) -> None:
        self.store[agent_name] = result
        self.logs.append({
            "agent": agent_name,
            "timestamp": str(datetime.now()),
            "status": "updated"
        })

    def get(self, agent_name: str) -> Optional[Any]:
        return self.store.get(agent_name)

    def get_all(self) -> Dict[str, Any]:
        return self.store

    def get_logs(self) -> List[Dict[str, Any]]:
        return self.logs


# ============================================================
# BASE AGENT
# ============================================================
class BaseAgent:
    def __init__(self, name: str, llm: LLMService):
        self.name = name
        self.llm = llm

    def save_to_memory(self, memory: SharedMemory, result: Any) -> Any:
        memory.update(self.name, result)
        return result


# ============================================================
# ORCHESTRATOR AGENT
# ============================================================
class OrchestratorAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Orchestrator Agent] Planning the EV research workflow...")

        system_prompt = (
            "You are the Orchestrator Agent in a multi-agent EV research pipeline. "
            "The domain is EV industry trends in India for 2025. "
            "Return JSON with keys: goal_summary, task_breakdown, execution_order, orchestration_note, expected_deliverable."
        )

        user_prompt = f"""
Research Goal:
{goal}

Return JSON only.
"""

        parsed = safe_json_parse(self.llm.generate(system_prompt, user_prompt, temperature=0.2))
        return self.save_to_memory(memory, parsed)


# ============================================================
# SEARCH AGENT
# ============================================================
class SearchAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Search Agent] Gathering EV market data, signals, and trend inputs...")

        orchestrator_plan = memory.get("orchestrator")

        system_prompt = (
            "You are the Search Agent in a multi-agent EV market research pipeline. "
            "Your role is to produce structured EV industry findings for India in 2025. "
            "Keep values concise and realistic. "
            "Return JSON with keys: "
            "industry_overview, key_trends, consumer_preferences, market_drivers, market_challenges, "
            "competitive_landscape, charging_infrastructure_outlook, policy_and_regulation_signals, forecast_signals, source_style_note."
        )

        user_prompt = f"""
Research Goal:
{goal}

Orchestrator Plan:
{json.dumps(orchestrator_plan, indent=2)}

Important:
- Keep each list between 3 and 5 items
- Keep text concise
- Return one JSON object only
"""

        parsed = safe_json_parse(self.llm.generate(system_prompt, user_prompt, temperature=0.2))
        return self.save_to_memory(memory, parsed)


# ============================================================
# ANALYST AGENT
# ============================================================
class AnalystAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Analyst Agent] Interpreting EV research data into strategic insights...")

        search_data = memory.get("search")

        system_prompt = (
            "You are the Analyst Agent in a multi-agent EV research pipeline. "
            "Turn EV findings into strategy insights. "
            "Return JSON with keys: executive_insights, opportunity_areas, risk_factors, "
            "strategic_interpretation, competitive_implications, recommended_focus_areas, confidence_assessment, analyst_note."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Return one JSON object only.
"""

        parsed = safe_json_parse(self.llm.generate(system_prompt, user_prompt, temperature=0.25))
        return self.save_to_memory(memory, parsed)


# ============================================================
# WRITER AGENT
# ============================================================
class WriterAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[Writer Agent] Drafting the EV market report in a structured professional format...")

        search_data = memory.get("search")
        analysis_data = memory.get("analyst")

        system_prompt = (
            "You are the Writer Agent in a multi-agent EV research pipeline. "
            "Write a management-grade report on EV industry trends in India for 2025. "
            "Return JSON with keys: report_title, executive_summary, report_sections, draft_report, writer_note."
        )

        user_prompt = f"""
Research Goal:
{goal}

Search Findings:
{json.dumps(search_data, indent=2)}

Analysis Insights:
{json.dumps(analysis_data, indent=2)}

Return one JSON object only.
"""

        parsed = safe_json_parse(self.llm.generate(system_prompt, user_prompt, temperature=0.35))
        return self.save_to_memory(memory, parsed)


# ============================================================
# DYNAMIC QA AGENT
# ============================================================
class QAAgent(BaseAgent):
    def run(self, goal: str, memory: SharedMemory) -> Dict[str, Any]:
        print("[QA Agent] Running dynamic review: quality, coverage, strategy value, tone, and revision upgrade...")

        writer_output = memory.get("writer")
        analyst_output = memory.get("analyst")
        search_output = memory.get("search")
        orchestrator_output = memory.get("orchestrator")

        system_prompt = (
            "You are the QA Agent in a multi-agent EV research pipeline. "
            "You must evaluate the draft for EV relevance, completeness, strategic usefulness, clarity, and executive readiness. "
            "Then produce a stronger revised version. "
            "Return JSON with keys: quality_score, quality_dimensions, missing_elements, improvement_actions, "
            "revised_executive_summary, final_report, quality_status, qa_note."
        )

        user_prompt = f"""
Research Goal:
{goal}

Orchestrator Output:
{json.dumps(orchestrator_output, indent=2)}

Search Output:
{json.dumps(search_output, indent=2)}

Analyst Output:
{json.dumps(analyst_output, indent=2)}

Writer Draft:
{json.dumps(writer_output, indent=2)}

Important:
- quality_score should be between 1 and 10
- quality_dimensions should include clarity, completeness, strategic_value, EV_relevance, executive_readability
- final_report should be improved and more polished than the draft
- return one JSON object only
"""

        parsed = safe_json_parse(self.llm.generate(system_prompt, user_prompt, temperature=0.2))
        return self.save_to_memory(memory, parsed)


# ============================================================
# MAIN PIPELINE
# ============================================================
class WorldClassEVResearchPipeline:
    def __init__(self, llm_service: LLMService):
        self.memory = SharedMemory()
        self.orchestrator = OrchestratorAgent("orchestrator", llm_service)
        self.search = SearchAgent("search", llm_service)
        self.analyst = AnalystAgent("analyst", llm_service)
        self.writer = WriterAgent("writer", llm_service)
        self.qa = QAAgent("qa", llm_service)

    def run(self, goal: str) -> Dict[str, Any]:
        print("=" * 90)
        print("WORLD-CLASS MULTI-AGENT AI RESEARCH TEAM")
        print("=" * 90)
        print("Goal:", goal)
        print("=" * 90)

        self.orchestrator.run(goal, self.memory)
        self.search.run(goal, self.memory)
        self.analyst.run(goal, self.memory)
        self.writer.run(goal, self.memory)
        self.qa.run(goal, self.memory)

        return {
            "memory": self.memory.get_all(),
            "logs": self.memory.get_logs()
        }


# ============================================================
# RUN PROJECT
# ============================================================
if __name__ == "__main__":
    llm_service = LLMService(
        api_key=GROQ_API_KEY,
        model=GROQ_MODEL
    )

    goal = "Write a comprehensive market report on EV industry trends in India for 2025."

    pipeline = WorldClassEVResearchPipeline(llm_service)
    result = pipeline.run(goal)

    final_qa_output = result["memory"]["qa"]

    print("\n" + "=" * 90)
    print("FINAL REVISED REPORT")
    print("=" * 90)
    print(final_qa_output["final_report"])

    print("\n" + "=" * 90)
    print("QUALITY SCORE")
    print("=" * 90)
    print(final_qa_output["quality_score"])

    print("\n" + "=" * 90)
    print("QUALITY DIMENSIONS")
    print("=" * 90)
    print(json.dumps(final_qa_output["quality_dimensions"], indent=2))

    print("\n" + "=" * 90)
    print("MISSING ELEMENTS DETECTED")
    print("=" * 90)
    print(json.dumps(final_qa_output["missing_elements"], indent=2))

    print("\n" + "=" * 90)
    print("IMPROVEMENT ACTIONS TAKEN")
    print("=" * 90)
    print(json.dumps(final_qa_output["improvement_actions"], indent=2))

    print("\n" + "=" * 90)
    print("PIPELINE LOGS")
    print("=" * 90)
    print(json.dumps(result["logs"], indent=2))

WORLD-CLASS MULTI-AGENT AI RESEARCH TEAM
Goal: Write a comprehensive market report on EV industry trends in India for 2025.
[Orchestrator Agent] Planning the EV research workflow...
[Search Agent] Gathering EV market data, signals, and trend inputs...
[Analyst Agent] Interpreting EV research data into strategic insights...
[Writer Agent] Drafting the EV market report in a structured professional format...
[QA Agent] Running dynamic review: quality, coverage, strategy value, tone, and revision upgrade...

FINAL REVISED REPORT
The Indian electric vehicle market is experiencing rapid growth, driven by increasing adoption, government incentives, and improving infrastructure. Key trends include increasing adoption, government incentives, and improving infrastructure, with consumer preferences focused on affordability, range anxiety, and brand reputation. Market drivers include government policies, environmental concerns, and decreasing battery costs, while market challenges such as limited 